In [17]:
pip install lingua-language-detector

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.5/172.5 MB 1.1 MB/s  0:01:59 eta 0:00:010:00:05m
Note: you may need to restart the kernel to use updated packages.


In [1]:
import sys, subprocess
for pkg in ["selenium", "pandas", "openpyxl", "langdetect"]:
    try:
        __import__(pkg.replace("-","_"))
        print(f"✅ {pkg}")
    except ImportError:
        subprocess.run([sys.executable,"-m","pip","install",pkg], check=True)
        print(f"✅ {pkg} installé")
print("Prêt.")

✅ selenium
✅ pandas
✅ openpyxl
✅ langdetect
Prêt.


In [1]:
import time
import re
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from lingua import Language, LanguageDetectorBuilder

# ==================================================
URL = "https://www.booking.com/hotel/ma/ibis-moussafir-casablanca-city-center.html?auth_success=1#tab-reviews"
HOTEL_NAME = "Ibis Casablanca City Center"
HOTEL_CITY = "Casablanca"
MAX_PAGES = 400
# ==================================================


class BookingScraper:
    def __init__(self):
        options = Options()
        options.add_argument('--lang=en')
        options.add_argument('--disable-blink-features=AutomationControlled')
        options.add_experimental_option("excludeSwitches", ["enable-automation"])
        options.add_experimental_option('useAutomationExtension', False)
        options.add_argument(
            '--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
            'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
        )
        self.driver = webdriver.Chrome(
            service=Service(ChromeDriverManager().install()), options=options
        )
        self.wait = WebDriverWait(self.driver, 20)

        self.detector = LanguageDetectorBuilder.from_languages(
            Language.FRENCH, Language.ENGLISH, Language.GERMAN,
            Language.SPANISH, Language.ITALIAN, Language.PORTUGUESE,
            Language.DUTCH, Language.RUSSIAN, Language.ARABIC, Language.CHINESE,
        ).with_minimum_relative_distance(0.15).build()

        self._lang_map = {
            Language.FRENCH: "french", Language.ENGLISH: "english",
            Language.GERMAN: "german", Language.SPANISH: "spanish",
            Language.ITALIAN: "italian", Language.PORTUGUESE: "portuguese",
            Language.DUTCH: "dutch", Language.RUSSIAN: "russian",
            Language.ARABIC: "arabic", Language.CHINESE: "chinese",
        }

    def _text(self, parent, css):
        try:
            return parent.find_element(By.CSS_SELECTOR, css).text.strip()
        except Exception:
            return ""

    def _first_text(self, parent, *selectors):
        for sel in selectors:
            val = self._text(parent, sel)
            if val:
                return val
        return ""

    def _extract_number(self, raw):
        if not raw:
            return ""
        raw = raw.replace(",", ".")
        match = re.search(r'\d+(?:\.\d+)?', raw)
        return match.group(0) if match else ""

    def detect_language(self, text):
        if not text or len(text.strip()) < 5:
            return "unknown"
        try:
            lang = self.detector.detect_language_of(text)
            return self._lang_map.get(lang, "unknown") if lang else "unknown"
        except Exception:
            return "unknown"

    def _get_snapshot(self):
        try:
            card = self.driver.find_element(By.CSS_SELECTOR, "div[data-testid='review-card']")
            for sel in [
                "div[data-testid='review-positive-text']",
                "div[data-testid='review-negative-text']",
                "h4[data-testid='review-title']",
            ]:
                try:
                    t = card.find_element(By.CSS_SELECTOR, sel).text.strip()
                    if t:
                        return t
                except Exception:
                    pass
        except Exception:
            pass
        return ""

    def _next_page(self, snapshot_before):
        MAX_RETRIES = 3

        for attempt in range(1, MAX_RETRIES + 1):
            btn = None

            labels = ["Page suivante", "Suivant", "Next page", "Next"]
            for label in labels:
                try:
                    btn = self.driver.find_element(
                        By.CSS_SELECTOR, f"button[aria-label='{label}']"
                    )
                    print(f"      🔘 Bouton trouvé : '{label}'")
                    break
                except Exception:
                    pass

            if btn is None:
                try:
                    btn = self.driver.find_element(
                        By.XPATH,
                        "//button[contains(@aria-label,'suivante') or "
                        "contains(@aria-label,'Suivant') or "
                        "contains(@aria-label,'Next')]"
                    )
                    print(f"      🔘 Bouton via XPath : '{btn.get_attribute('aria-label')}'")
                except Exception:
                    pass

            if btn is None:
                print("   ⚠️  Bouton introuvable.")
                return False

            disabled      = btn.get_attribute("disabled")
            aria_disabled = btn.get_attribute("aria-disabled") or ""
            classes       = btn.get_attribute("class") or ""
            if disabled or aria_disabled == "true" or "disabled" in classes:
                print("   ✅ Bouton désactivé — dernière page atteinte.")
                return False

            # Scroll vers le bouton
            try:
                self.driver.execute_script(
                    "arguments[0].scrollIntoView({block:'center'});", btn
                )
                time.sleep(1)
            except Exception:
                pass

            # Clic
            clicked = False
            for method in ['normal', 'js']:
                try:
                    if method == 'normal':
                        btn.click()
                    else:
                        self.driver.execute_script("arguments[0].click();", btn)
                    clicked = True
                    print(f"      ✅ Clic OK ({method}) — tentative {attempt}/{MAX_RETRIES}")
                    break
                except Exception as e:
                    print(f"      ⚠️  Clic {method} échoué : {e}")

            if not clicked:
                time.sleep(2)
                continue

            # Poll 30s
            for _ in range(60):
                time.sleep(0.5)
                try:
                    cards = self.driver.find_elements(
                        By.CSS_SELECTOR, "div[data-testid='review-card']"
                    )
                    if not cards:
                        continue
                    first_text = ""
                    for sel in [
                        "div[data-testid='review-positive-text']",
                        "div[data-testid='review-negative-text']",
                        "h4[data-testid='review-title']",
                    ]:
                        try:
                            t = cards[0].find_element(By.CSS_SELECTOR, sel).text.strip()
                            if t:
                                first_text = t
                                break
                        except Exception:
                            pass
                    if first_text and first_text != snapshot_before:
                        return True
                except Exception:
                    pass

            print(f"   ⚠️  Contenu inchangé après 30s (tentative {attempt}/{MAX_RETRIES})")

            if attempt < MAX_RETRIES:
                print(f"   🔄 Nouvelle tentative dans 3s...")
                self.driver.execute_script("window.scrollTo(0, 0);")
                time.sleep(3)

        print("   ❌ Échec après 3 tentatives.")
        return False

    def _parse_card(self, card, hotel_name, hotel_city):
        name = self._first_text(
            card,
            "div.b08850ce41.f546354b44",
            "div[data-testid='reviewer-name'] span",
            "p.reviewer_name",
        )
        nationality = self._first_text(
            card,
            "span.d838fb5f41.aea5eccb71",
            "span[class*='reviewer_country']",
            "div[data-testid='reviewer-country'] span",
        )
        raw_rating = self._first_text(
            card,
            "div[data-testid='review-score'] div.bc946a29db",
            "div[data-testid='review-score']",
            "div.bui-review-score__badge",
        )
        rating    = self._extract_number(raw_rating)
        title     = self._first_text(card, "h4[data-testid='review-title']", "h3[class*='title']")
        pub_date  = self._first_text(card, "span[data-testid='review-date']", "p[data-testid='review-date']")
        pub_date  = re.sub(r'^Reviewed\s*:\s*', '', pub_date).strip()
        pub_date  = re.sub(r'^Rédigé le\s*:\s*', '', pub_date).strip()
        stay_date = self._first_text(card, "span[data-testid='review-stay-date']", "p.review_staydate")
        stay_date = re.sub(r'^Stayed\s*:\s*', '', stay_date).strip()
        stay_date = re.sub(r'^Séjour\s*:\s*', '', stay_date).strip()
        trip_type = self._first_text(
            card,
            "span[data-testid='review-traveler-type']",
            "p[data-testid='travel-type']",
            "div[data-testid='review-travel-type']",
        )
        pos  = self._text(card, "div[data-testid='review-positive-text']")
        neg  = self._text(card, "div[data-testid='review-negative-text']")
        full = (pos + " " + neg).strip()

        sub_scores = {}
        try:
            subs = card.find_elements(By.CSS_SELECTOR, "div[data-testid='review-subscore']")
            for s in subs:
                try:
                    cat = s.find_element(By.CSS_SELECTOR, "span").text.strip()
                    val = s.find_element(By.CSS_SELECTOR, "div.bc946a29db").text.strip()
                    sub_scores[cat] = self._extract_number(val)
                except Exception:
                    pass
        except Exception:
            pass

        return {
            "source": "booking.com",
            "hotel_name": hotel_name,
            "hotel_city": hotel_city,
            "reviewer_name": name,
            "nationality": nationality,
            "rating": rating,
            "review_title": title,
            "review_text": full,
            "positive_comment": pos,
            "negative_comment": neg,
            "publication_date": pub_date,
            "stay_date": stay_date,
            "trip_type": trip_type,
            "language": self.detect_language(full),
            **sub_scores,
        }

    def scrape_reviews(self, url, hotel_name, hotel_city, max_pages=400):
        print(f"🔄 Scraping : {hotel_name} ({hotel_city})")
        self.driver.get(url)
        time.sleep(6)

        all_reviews = []
        seen = set()
        page = 1

        while page <= max_pages:
            print(f"   📄 Page {page}  ({len(all_reviews)} avis cumulés)")

            try:
                self.wait.until(
                    EC.presence_of_element_located(
                        (By.CSS_SELECTOR, "div[data-testid='review-card']")
                    )
                )
            except Exception:
                print("   ⚠️  Aucune carte trouvée, fin du scraping.")
                break

            cards = self.driver.find_elements(
                By.CSS_SELECTOR, "div[data-testid='review-card']"
            )
            if not cards:
                print("   ⚠️  Liste de cartes vide, fin du scraping.")
                break

            snap = self._get_snapshot()

            new_count = 0
            for card in cards:
                review = self._parse_card(card, hotel_name, hotel_city)
                key = (
                    review.get("reviewer_name", ""),
                    review.get("publication_date", ""),
                    review.get("rating", ""),
                    review.get("review_text", "")[:50],
                )
                if key not in seen:
                    seen.add(key)
                    all_reviews.append(review)
                    new_count += 1

            print(f"      → {new_count} nouveaux avis")

            if new_count == 0:
                print("   ⚠️  Aucun nouvel avis — boucle détectée, arrêt.")
                break

            if not self._next_page(snap):
                print("   ✅ Dernière page atteinte.")
                break

            page += 1

        self.driver.quit()
        print(f"\n✅ {len(all_reviews)} avis extraits au total")
        return pd.DataFrame(all_reviews)


# ==================================================
if __name__ == "__main__":
    scraper = BookingScraper()
    df = scraper.scrape_reviews(URL, HOTEL_NAME, HOTEL_CITY, max_pages=MAX_PAGES)

    if not df.empty:
        filename = f"avis_{HOTEL_NAME.replace(' ', '_')}.csv"
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"\n💾 Fichier sauvegardé : {filename}  ({len(df)} avis)")

        print("\n🔍 Aperçu :")
        cols = [c for c in ["reviewer_name", "nationality", "rating",
                             "trip_type", "publication_date", "stay_date",
                             "language"] if c in df.columns]
        print(df[cols].head(10).to_string())

        print("\n📊 Colonnes vides :")
        empty = df[cols].isnull().sum() + (df[cols] == "").sum()
        print(empty[empty > 0] if empty[empty > 0].any() else "  Aucune colonne vide 🎉")
    else:
        print("❌ Aucun avis extrait.")

🔄 Scraping : Ibis Casablanca City Center (Casablanca)
   📄 Page 1  (0 avis cumulés)
      → 9 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 2  (9 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 3  (19 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 4  (29 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 5  (39 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 6  (49 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 7  (59 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tent

In [2]:
df.head()

,source,hotel_name,hotel_city,reviewer_name,nationality,rating,review_title,review_text,positive_comment,negative_comment,publication_date,stay_date,trip_type,language
0,booking.com,Ibis Casablanca City Center,Casablanca,Hermann,France,8.0,Très bon emplacement pour visiter Casablanca,Super emplacement. Personnel très sympathique....,Super emplacement. Personnel très sympathique....,"Oreillers à changer, car déformés. Je pense qu...",Commentaire envoyé le 20 avril 2026,avril 2026,Famille,french
1,booking.com,Ibis Casablanca City Center,Casablanca,Linda,Espagne,8.0,Séjour Casablanca,Personnel à la réception très courtois et atte...,Personnel à la réception très courtois et atte...,Manque de fruits et fromage au petit déjeuner,Commentaire envoyé le 12 avril 2026,avril 2026,Voyageur individuel,french
2,booking.com,Ibis Casablanca City Center,Casablanca,Ifriquine,Maroc,9.0,Fabuleux,Le personnel est tres sympa Rien,Le personnel est tres sympa,Rien,Commentaire envoyé le 9 avril 2026,avril 2026,Famille,french
3,booking.com,Ibis Casablanca City Center,Casablanca,Mohamed,Maroc,7.0,Bon séjour,Bon séjour rapide RAS,Bon séjour rapide,RAS,Commentaire envoyé le 9 avril 2026,avril 2026,Voyageur individuel,french
4,booking.com,Ibis Casablanca City Center,Casablanca,Aboubakr,France,9.0,sympa,la proximité de tous le prix la propreté \nle ...,la proximité de tous le prix la propreté \nle ...,le parquet de la chambre,Commentaire envoyé le 7 avril 2026,avril 2026,Voyageur individuel,french


In [4]:
df.shape

(4000, 14)

In [3]:
import time
import re
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from lingua import Language, LanguageDetectorBuilder

# ==================================================
URL = "https://www.booking.com/hotel/ma/ibis-casablanca-sidi-maarouf.html?auth_success=1#tab-reviews"
HOTEL_NAME = "Ibis Casablanca Sidi Maarouf"
HOTEL_CITY = "Casablanca"
MAX_PAGES = 90
# ==================================================


class BookingScraper:
    def __init__(self):
        options = Options()
        options.add_argument('--lang=en')
        options.add_argument('--disable-blink-features=AutomationControlled')
        options.add_experimental_option("excludeSwitches", ["enable-automation"])
        options.add_experimental_option('useAutomationExtension', False)
        options.add_argument(
            '--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
            'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
        )
        self.driver = webdriver.Chrome(
            service=Service(ChromeDriverManager().install()), options=options
        )
        self.wait = WebDriverWait(self.driver, 20)

        self.detector = LanguageDetectorBuilder.from_languages(
            Language.FRENCH, Language.ENGLISH, Language.GERMAN,
            Language.SPANISH, Language.ITALIAN, Language.PORTUGUESE,
            Language.DUTCH, Language.RUSSIAN, Language.ARABIC, Language.CHINESE,
        ).with_minimum_relative_distance(0.15).build()

        self._lang_map = {
            Language.FRENCH: "french", Language.ENGLISH: "english",
            Language.GERMAN: "german", Language.SPANISH: "spanish",
            Language.ITALIAN: "italian", Language.PORTUGUESE: "portuguese",
            Language.DUTCH: "dutch", Language.RUSSIAN: "russian",
            Language.ARABIC: "arabic", Language.CHINESE: "chinese",
        }

    def _text(self, parent, css):
        try:
            return parent.find_element(By.CSS_SELECTOR, css).text.strip()
        except Exception:
            return ""

    def _first_text(self, parent, *selectors):
        for sel in selectors:
            val = self._text(parent, sel)
            if val:
                return val
        return ""

    def _extract_number(self, raw):
        if not raw:
            return ""
        raw = raw.replace(",", ".")
        match = re.search(r'\d+(?:\.\d+)?', raw)
        return match.group(0) if match else ""

    def detect_language(self, text):
        if not text or len(text.strip()) < 5:
            return "unknown"
        try:
            lang = self.detector.detect_language_of(text)
            return self._lang_map.get(lang, "unknown") if lang else "unknown"
        except Exception:
            return "unknown"

    def _get_snapshot(self):
        try:
            card = self.driver.find_element(By.CSS_SELECTOR, "div[data-testid='review-card']")
            for sel in [
                "div[data-testid='review-positive-text']",
                "div[data-testid='review-negative-text']",
                "h4[data-testid='review-title']",
            ]:
                try:
                    t = card.find_element(By.CSS_SELECTOR, sel).text.strip()
                    if t:
                        return t
                except Exception:
                    pass
        except Exception:
            pass
        return ""

    def _next_page(self, snapshot_before):
        MAX_RETRIES = 3

        for attempt in range(1, MAX_RETRIES + 1):
            btn = None

            labels = ["Page suivante", "Suivant", "Next page", "Next"]
            for label in labels:
                try:
                    btn = self.driver.find_element(
                        By.CSS_SELECTOR, f"button[aria-label='{label}']"
                    )
                    print(f"      🔘 Bouton trouvé : '{label}'")
                    break
                except Exception:
                    pass

            if btn is None:
                try:
                    btn = self.driver.find_element(
                        By.XPATH,
                        "//button[contains(@aria-label,'suivante') or "
                        "contains(@aria-label,'Suivant') or "
                        "contains(@aria-label,'Next')]"
                    )
                    print(f"      🔘 Bouton via XPath : '{btn.get_attribute('aria-label')}'")
                except Exception:
                    pass

            if btn is None:
                print("   ⚠️  Bouton introuvable.")
                return False

            disabled      = btn.get_attribute("disabled")
            aria_disabled = btn.get_attribute("aria-disabled") or ""
            classes       = btn.get_attribute("class") or ""
            if disabled or aria_disabled == "true" or "disabled" in classes:
                print("   ✅ Bouton désactivé — dernière page atteinte.")
                return False

            try:
                self.driver.execute_script(
                    "arguments[0].scrollIntoView({block:'center'});", btn
                )
                time.sleep(1)
            except Exception:
                pass

            clicked = False
            for method in ['normal', 'js']:
                try:
                    if method == 'normal':
                        btn.click()
                    else:
                        self.driver.execute_script("arguments[0].click();", btn)
                    clicked = True
                    print(f"      ✅ Clic OK ({method}) — tentative {attempt}/{MAX_RETRIES}")
                    break
                except Exception as e:
                    print(f"      ⚠️  Clic {method} échoué : {e}")

            if not clicked:
                time.sleep(2)
                continue

            for _ in range(60):
                time.sleep(0.5)
                try:
                    cards = self.driver.find_elements(
                        By.CSS_SELECTOR, "div[data-testid='review-card']"
                    )
                    if not cards:
                        continue
                    first_text = ""
                    for sel in [
                        "div[data-testid='review-positive-text']",
                        "div[data-testid='review-negative-text']",
                        "h4[data-testid='review-title']",
                    ]:
                        try:
                            t = cards[0].find_element(By.CSS_SELECTOR, sel).text.strip()
                            if t:
                                first_text = t
                                break
                        except Exception:
                            pass
                    if first_text and first_text != snapshot_before:
                        return True
                except Exception:
                    pass

            print(f"   ⚠️  Contenu inchangé après 30s (tentative {attempt}/{MAX_RETRIES})")

            if attempt < MAX_RETRIES:
                print(f"   🔄 Nouvelle tentative dans 3s...")
                self.driver.execute_script("window.scrollTo(0, 0);")
                time.sleep(3)

        print("   ❌ Échec après 3 tentatives.")
        return False

    def _parse_card(self, card, hotel_name, hotel_city):
        name = self._first_text(
            card,
            "div.b08850ce41.f546354b44",
            "div[data-testid='reviewer-name'] span",
            "p.reviewer_name",
        )
        nationality = self._first_text(
            card,
            "span.d838fb5f41.aea5eccb71",
            "span[class*='reviewer_country']",
            "div[data-testid='reviewer-country'] span",
        )
        raw_rating = self._first_text(
            card,
            "div[data-testid='review-score'] div.bc946a29db",
            "div[data-testid='review-score']",
            "div.bui-review-score__badge",
        )
        rating    = self._extract_number(raw_rating)
        title     = self._first_text(card, "h4[data-testid='review-title']", "h3[class*='title']")
        pub_date  = self._first_text(card, "span[data-testid='review-date']", "p[data-testid='review-date']")
        pub_date  = re.sub(r'^Reviewed\s*:\s*', '', pub_date).strip()
        pub_date  = re.sub(r'^Rédigé le\s*:\s*', '', pub_date).strip()
        stay_date = self._first_text(card, "span[data-testid='review-stay-date']", "p.review_staydate")
        stay_date = re.sub(r'^Stayed\s*:\s*', '', stay_date).strip()
        stay_date = re.sub(r'^Séjour\s*:\s*', '', stay_date).strip()
        trip_type = self._first_text(
            card,
            "span[data-testid='review-traveler-type']",
            "p[data-testid='travel-type']",
            "div[data-testid='review-travel-type']",
        )
        pos  = self._text(card, "div[data-testid='review-positive-text']")
        neg  = self._text(card, "div[data-testid='review-negative-text']")
        full = (pos + " " + neg).strip()

        sub_scores = {}
        try:
            subs = card.find_elements(By.CSS_SELECTOR, "div[data-testid='review-subscore']")
            for s in subs:
                try:
                    cat = s.find_element(By.CSS_SELECTOR, "span").text.strip()
                    val = s.find_element(By.CSS_SELECTOR, "div.bc946a29db").text.strip()
                    sub_scores[cat] = self._extract_number(val)
                except Exception:
                    pass
        except Exception:
            pass

        return {
            "source": "booking.com",
            "hotel_name": hotel_name,
            "hotel_city": hotel_city,
            "reviewer_name": name,
            "nationality": nationality,
            "rating": rating,
            "review_title": title,
            "review_text": full,
            "positive_comment": pos,
            "negative_comment": neg,
            "publication_date": pub_date,
            "stay_date": stay_date,
            "trip_type": trip_type,
            "language": self.detect_language(full),
            **sub_scores,
        }

    def scrape_reviews(self, url, hotel_name, hotel_city, max_pages=90):
        print(f"🔄 Scraping : {hotel_name} ({hotel_city})")
        self.driver.get(url)
        time.sleep(6)

        all_reviews = []
        seen = set()
        page = 1

        while page <= max_pages:
            print(f"   📄 Page {page}  ({len(all_reviews)} avis cumulés)")

            try:
                self.wait.until(
                    EC.presence_of_element_located(
                        (By.CSS_SELECTOR, "div[data-testid='review-card']")
                    )
                )
            except Exception:
                print("   ⚠️  Aucune carte trouvée, fin du scraping.")
                break

            cards = self.driver.find_elements(
                By.CSS_SELECTOR, "div[data-testid='review-card']"
            )
            if not cards:
                print("   ⚠️  Liste de cartes vide, fin du scraping.")
                break

            snap = self._get_snapshot()

            new_count = 0
            for card in cards:
                review = self._parse_card(card, hotel_name, hotel_city)
                key = (
                    review.get("reviewer_name", ""),
                    review.get("publication_date", ""),
                    review.get("rating", ""),
                    review.get("review_text", "")[:50],
                )
                if key not in seen:
                    seen.add(key)
                    all_reviews.append(review)
                    new_count += 1

            print(f"      → {new_count} nouveaux avis")

            if new_count == 0:
                print("   ⚠️  Aucun nouvel avis — boucle détectée, arrêt.")
                break

            if not self._next_page(snap):
                print("   ✅ Dernière page atteinte.")
                break

            page += 1

        self.driver.quit()
        print(f"\n✅ {len(all_reviews)} avis extraits au total")
        return pd.DataFrame(all_reviews)


# ==================================================
if __name__ == "__main__":
    scraper = BookingScraper()
    df = scraper.scrape_reviews(URL, HOTEL_NAME, HOTEL_CITY, max_pages=MAX_PAGES)

    if not df.empty:
        filename = f"avis_{HOTEL_NAME.replace(' ', '_')}.csv"
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"\n💾 Fichier sauvegardé : {filename}  ({len(df)} avis)")

        print("\n🔍 Aperçu :")
        cols = [c for c in ["reviewer_name", "nationality", "rating",
                             "trip_type", "publication_date", "stay_date",
                             "language"] if c in df.columns]
        print(df[cols].head(10).to_string())

        print("\n📊 Colonnes vides :")
        empty = df[cols].isnull().sum() + (df[cols] == "").sum()
        print(empty[empty > 0] if empty[empty > 0].any() else "  Aucune colonne vide 🎉")
    else:
        print("❌ Aucun avis extrait.")

🔄 Scraping : Ibis Casablanca Sidi Maarouf (Casablanca)
   📄 Page 1  (0 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 2  (10 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 3  (20 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 4  (30 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 5  (40 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 6  (50 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 7  (60 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — t

In [4]:
import time
import re
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from lingua import Language, LanguageDetectorBuilder

# ==================================================
URL = "https://www.booking.com/hotel/ma/ibis-abdelmoumen.html?auth_success=1#tab-reviews"
HOTEL_NAME = "Ibis Casablanca Abdelmoumen"
HOTEL_CITY = "Casablanca"
MAX_PAGES = 300
# ==================================================


class BookingScraper:
    def __init__(self):
        options = Options()
        options.add_argument('--lang=en')
        options.add_argument('--disable-blink-features=AutomationControlled')
        options.add_experimental_option("excludeSwitches", ["enable-automation"])
        options.add_experimental_option('useAutomationExtension', False)
        options.add_argument(
            '--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
            'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
        )
        self.driver = webdriver.Chrome(
            service=Service(ChromeDriverManager().install()), options=options
        )
        self.wait = WebDriverWait(self.driver, 20)

        self.detector = LanguageDetectorBuilder.from_languages(
            Language.FRENCH, Language.ENGLISH, Language.GERMAN,
            Language.SPANISH, Language.ITALIAN, Language.PORTUGUESE,
            Language.DUTCH, Language.RUSSIAN, Language.ARABIC, Language.CHINESE,
        ).with_minimum_relative_distance(0.15).build()

        self._lang_map = {
            Language.FRENCH: "french", Language.ENGLISH: "english",
            Language.GERMAN: "german", Language.SPANISH: "spanish",
            Language.ITALIAN: "italian", Language.PORTUGUESE: "portuguese",
            Language.DUTCH: "dutch", Language.RUSSIAN: "russian",
            Language.ARABIC: "arabic", Language.CHINESE: "chinese",
        }

    def _text(self, parent, css):
        try:
            return parent.find_element(By.CSS_SELECTOR, css).text.strip()
        except Exception:
            return ""

    def _first_text(self, parent, *selectors):
        for sel in selectors:
            val = self._text(parent, sel)
            if val:
                return val
        return ""

    def _extract_number(self, raw):
        if not raw:
            return ""
        raw = raw.replace(",", ".")
        match = re.search(r'\d+(?:\.\d+)?', raw)
        return match.group(0) if match else ""

    def detect_language(self, text):
        if not text or len(text.strip()) < 5:
            return "unknown"
        try:
            lang = self.detector.detect_language_of(text)
            return self._lang_map.get(lang, "unknown") if lang else "unknown"
        except Exception:
            return "unknown"

    def _get_snapshot(self):
        try:
            card = self.driver.find_element(By.CSS_SELECTOR, "div[data-testid='review-card']")
            for sel in [
                "div[data-testid='review-positive-text']",
                "div[data-testid='review-negative-text']",
                "h4[data-testid='review-title']",
            ]:
                try:
                    t = card.find_element(By.CSS_SELECTOR, sel).text.strip()
                    if t:
                        return t
                except Exception:
                    pass
        except Exception:
            pass
        return ""

    def _next_page(self, snapshot_before):
        MAX_RETRIES = 3

        for attempt in range(1, MAX_RETRIES + 1):
            btn = None

            labels = ["Page suivante", "Suivant", "Next page", "Next"]
            for label in labels:
                try:
                    btn = self.driver.find_element(
                        By.CSS_SELECTOR, f"button[aria-label='{label}']"
                    )
                    print(f"      🔘 Bouton trouvé : '{label}'")
                    break
                except Exception:
                    pass

            if btn is None:
                try:
                    btn = self.driver.find_element(
                        By.XPATH,
                        "//button[contains(@aria-label,'suivante') or "
                        "contains(@aria-label,'Suivant') or "
                        "contains(@aria-label,'Next')]"
                    )
                    print(f"      🔘 Bouton via XPath : '{btn.get_attribute('aria-label')}'")
                except Exception:
                    pass

            if btn is None:
                print("   ⚠️  Bouton introuvable.")
                return False

            disabled      = btn.get_attribute("disabled")
            aria_disabled = btn.get_attribute("aria-disabled") or ""
            classes       = btn.get_attribute("class") or ""
            if disabled or aria_disabled == "true" or "disabled" in classes:
                print("   ✅ Bouton désactivé — dernière page atteinte.")
                return False

            try:
                self.driver.execute_script(
                    "arguments[0].scrollIntoView({block:'center'});", btn
                )
                time.sleep(1)
            except Exception:
                pass

            clicked = False
            for method in ['normal', 'js']:
                try:
                    if method == 'normal':
                        btn.click()
                    else:
                        self.driver.execute_script("arguments[0].click();", btn)
                    clicked = True
                    print(f"      ✅ Clic OK ({method}) — tentative {attempt}/{MAX_RETRIES}")
                    break
                except Exception as e:
                    print(f"      ⚠️  Clic {method} échoué : {e}")

            if not clicked:
                time.sleep(2)
                continue

            for _ in range(60):
                time.sleep(0.5)
                try:
                    cards = self.driver.find_elements(
                        By.CSS_SELECTOR, "div[data-testid='review-card']"
                    )
                    if not cards:
                        continue
                    first_text = ""
                    for sel in [
                        "div[data-testid='review-positive-text']",
                        "div[data-testid='review-negative-text']",
                        "h4[data-testid='review-title']",
                    ]:
                        try:
                            t = cards[0].find_element(By.CSS_SELECTOR, sel).text.strip()
                            if t:
                                first_text = t
                                break
                        except Exception:
                            pass
                    if first_text and first_text != snapshot_before:
                        return True
                except Exception:
                    pass

            print(f"   ⚠️  Contenu inchangé après 30s (tentative {attempt}/{MAX_RETRIES})")

            if attempt < MAX_RETRIES:
                print(f"   🔄 Nouvelle tentative dans 3s...")
                self.driver.execute_script("window.scrollTo(0, 0);")
                time.sleep(3)

        print("   ❌ Échec après 3 tentatives.")
        return False

    def _parse_card(self, card, hotel_name, hotel_city):
        name = self._first_text(
            card,
            "div.b08850ce41.f546354b44",
            "div[data-testid='reviewer-name'] span",
            "p.reviewer_name",
        )
        nationality = self._first_text(
            card,
            "span.d838fb5f41.aea5eccb71",
            "span[class*='reviewer_country']",
            "div[data-testid='reviewer-country'] span",
        )
        raw_rating = self._first_text(
            card,
            "div[data-testid='review-score'] div.bc946a29db",
            "div[data-testid='review-score']",
            "div.bui-review-score__badge",
        )
        rating    = self._extract_number(raw_rating)
        title     = self._first_text(card, "h4[data-testid='review-title']", "h3[class*='title']")
        pub_date  = self._first_text(card, "span[data-testid='review-date']", "p[data-testid='review-date']")
        pub_date  = re.sub(r'^Reviewed\s*:\s*', '', pub_date).strip()
        pub_date  = re.sub(r'^Rédigé le\s*:\s*', '', pub_date).strip()
        stay_date = self._first_text(card, "span[data-testid='review-stay-date']", "p.review_staydate")
        stay_date = re.sub(r'^Stayed\s*:\s*', '', stay_date).strip()
        stay_date = re.sub(r'^Séjour\s*:\s*', '', stay_date).strip()
        trip_type = self._first_text(
            card,
            "span[data-testid='review-traveler-type']",
            "p[data-testid='travel-type']",
            "div[data-testid='review-travel-type']",
        )
        pos  = self._text(card, "div[data-testid='review-positive-text']")
        neg  = self._text(card, "div[data-testid='review-negative-text']")
        full = (pos + " " + neg).strip()

        sub_scores = {}
        try:
            subs = card.find_elements(By.CSS_SELECTOR, "div[data-testid='review-subscore']")
            for s in subs:
                try:
                    cat = s.find_element(By.CSS_SELECTOR, "span").text.strip()
                    val = s.find_element(By.CSS_SELECTOR, "div.bc946a29db").text.strip()
                    sub_scores[cat] = self._extract_number(val)
                except Exception:
                    pass
        except Exception:
            pass

        return {
            "source": "booking.com",
            "hotel_name": hotel_name,
            "hotel_city": hotel_city,
            "reviewer_name": name,
            "nationality": nationality,
            "rating": rating,
            "review_title": title,
            "review_text": full,
            "positive_comment": pos,
            "negative_comment": neg,
            "publication_date": pub_date,
            "stay_date": stay_date,
            "trip_type": trip_type,
            "language": self.detect_language(full),
            **sub_scores,
        }

    def scrape_reviews(self, url, hotel_name, hotel_city, max_pages=90):
        print(f"🔄 Scraping : {hotel_name} ({hotel_city})")
        self.driver.get(url)
        time.sleep(6)

        all_reviews = []
        seen = set()
        page = 1

        while page <= max_pages:
            print(f"   📄 Page {page}  ({len(all_reviews)} avis cumulés)")

            try:
                self.wait.until(
                    EC.presence_of_element_located(
                        (By.CSS_SELECTOR, "div[data-testid='review-card']")
                    )
                )
            except Exception:
                print("   ⚠️  Aucune carte trouvée, fin du scraping.")
                break

            cards = self.driver.find_elements(
                By.CSS_SELECTOR, "div[data-testid='review-card']"
            )
            if not cards:
                print("   ⚠️  Liste de cartes vide, fin du scraping.")
                break

            snap = self._get_snapshot()

            new_count = 0
            for card in cards:
                review = self._parse_card(card, hotel_name, hotel_city)
                key = (
                    review.get("reviewer_name", ""),
                    review.get("publication_date", ""),
                    review.get("rating", ""),
                    review.get("review_text", "")[:50],
                )
                if key not in seen:
                    seen.add(key)
                    all_reviews.append(review)
                    new_count += 1

            print(f"      → {new_count} nouveaux avis")

            if new_count == 0:
                print("   ⚠️  Aucun nouvel avis — boucle détectée, arrêt.")
                break

            if not self._next_page(snap):
                print("   ✅ Dernière page atteinte.")
                break

            page += 1

        self.driver.quit()
        print(f"\n✅ {len(all_reviews)} avis extraits au total")
        return pd.DataFrame(all_reviews)


# ==================================================
if __name__ == "__main__":
    scraper = BookingScraper()
    df = scraper.scrape_reviews(URL, HOTEL_NAME, HOTEL_CITY, max_pages=MAX_PAGES)

    if not df.empty:
        filename = f"avis_{HOTEL_NAME.replace(' ', '_')}.csv"
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"\n💾 Fichier sauvegardé : {filename}  ({len(df)} avis)")

        print("\n🔍 Aperçu :")
        cols = [c for c in ["reviewer_name", "nationality", "rating",
                             "trip_type", "publication_date", "stay_date",
                             "language"] if c in df.columns]
        print(df[cols].head(10).to_string())

        print("\n📊 Colonnes vides :")
        empty = df[cols].isnull().sum() + (df[cols] == "").sum()
        print(empty[empty > 0] if empty[empty > 0].any() else "  Aucune colonne vide 🎉")
    else:
        print("❌ Aucun avis extrait.")

🔄 Scraping : Ibis Casablanca Abdelmoumen (Casablanca)
   📄 Page 1  (0 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 2  (10 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 3  (20 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 4  (30 avis cumulés)
      → 9 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 5  (39 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 6  (49 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 7  (59 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — ten

In [5]:
import time
import re
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from lingua import Language, LanguageDetectorBuilder

# ==================================================
URL = "https://www.booking.com/hotel/ma/ibis-casa-voyageurs.html?auth_success=1#tab-reviews"
HOTEL_NAME = "Ibis Casablanca Voyageurs"
HOTEL_CITY = "Casablanca"
MAX_PAGES = 300
# ==================================================


class BookingScraper:
    def __init__(self):
        options = Options()
        options.add_argument('--lang=en')
        options.add_argument('--disable-blink-features=AutomationControlled')
        options.add_experimental_option("excludeSwitches", ["enable-automation"])
        options.add_experimental_option('useAutomationExtension', False)
        options.add_argument(
            '--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
            'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
        )
        self.driver = webdriver.Chrome(
            service=Service(ChromeDriverManager().install()), options=options
        )
        self.wait = WebDriverWait(self.driver, 20)

        self.detector = LanguageDetectorBuilder.from_languages(
            Language.FRENCH, Language.ENGLISH, Language.GERMAN,
            Language.SPANISH, Language.ITALIAN, Language.PORTUGUESE,
            Language.DUTCH, Language.RUSSIAN, Language.ARABIC, Language.CHINESE,
        ).with_minimum_relative_distance(0.15).build()

        self._lang_map = {
            Language.FRENCH: "french", Language.ENGLISH: "english",
            Language.GERMAN: "german", Language.SPANISH: "spanish",
            Language.ITALIAN: "italian", Language.PORTUGUESE: "portuguese",
            Language.DUTCH: "dutch", Language.RUSSIAN: "russian",
            Language.ARABIC: "arabic", Language.CHINESE: "chinese",
        }

    def _text(self, parent, css):
        try:
            return parent.find_element(By.CSS_SELECTOR, css).text.strip()
        except Exception:
            return ""

    def _first_text(self, parent, *selectors):
        for sel in selectors:
            val = self._text(parent, sel)
            if val:
                return val
        return ""

    def _extract_number(self, raw):
        if not raw:
            return ""
        raw = raw.replace(",", ".")
        match = re.search(r'\d+(?:\.\d+)?', raw)
        return match.group(0) if match else ""

    def detect_language(self, text):
        if not text or len(text.strip()) < 5:
            return "unknown"
        try:
            lang = self.detector.detect_language_of(text)
            return self._lang_map.get(lang, "unknown") if lang else "unknown"
        except Exception:
            return "unknown"

    def _get_snapshot(self):
        try:
            card = self.driver.find_element(By.CSS_SELECTOR, "div[data-testid='review-card']")
            for sel in [
                "div[data-testid='review-positive-text']",
                "div[data-testid='review-negative-text']",
                "h4[data-testid='review-title']",
            ]:
                try:
                    t = card.find_element(By.CSS_SELECTOR, sel).text.strip()
                    if t:
                        return t
                except Exception:
                    pass
        except Exception:
            pass
        return ""

    def _next_page(self, snapshot_before):
        MAX_RETRIES = 3

        for attempt in range(1, MAX_RETRIES + 1):
            btn = None

            labels = ["Page suivante", "Suivant", "Next page", "Next"]
            for label in labels:
                try:
                    btn = self.driver.find_element(
                        By.CSS_SELECTOR, f"button[aria-label='{label}']"
                    )
                    print(f"      🔘 Bouton trouvé : '{label}'")
                    break
                except Exception:
                    pass

            if btn is None:
                try:
                    btn = self.driver.find_element(
                        By.XPATH,
                        "//button[contains(@aria-label,'suivante') or "
                        "contains(@aria-label,'Suivant') or "
                        "contains(@aria-label,'Next')]"
                    )
                    print(f"      🔘 Bouton via XPath : '{btn.get_attribute('aria-label')}'")
                except Exception:
                    pass

            if btn is None:
                print("   ⚠️  Bouton introuvable.")
                return False

            disabled      = btn.get_attribute("disabled")
            aria_disabled = btn.get_attribute("aria-disabled") or ""
            classes       = btn.get_attribute("class") or ""
            if disabled or aria_disabled == "true" or "disabled" in classes:
                print("   ✅ Bouton désactivé — dernière page atteinte.")
                return False

            try:
                self.driver.execute_script(
                    "arguments[0].scrollIntoView({block:'center'});", btn
                )
                time.sleep(1)
            except Exception:
                pass

            clicked = False
            for method in ['normal', 'js']:
                try:
                    if method == 'normal':
                        btn.click()
                    else:
                        self.driver.execute_script("arguments[0].click();", btn)
                    clicked = True
                    print(f"      ✅ Clic OK ({method}) — tentative {attempt}/{MAX_RETRIES}")
                    break
                except Exception as e:
                    print(f"      ⚠️  Clic {method} échoué : {e}")

            if not clicked:
                time.sleep(2)
                continue

            for _ in range(60):
                time.sleep(0.5)
                try:
                    cards = self.driver.find_elements(
                        By.CSS_SELECTOR, "div[data-testid='review-card']"
                    )
                    if not cards:
                        continue
                    first_text = ""
                    for sel in [
                        "div[data-testid='review-positive-text']",
                        "div[data-testid='review-negative-text']",
                        "h4[data-testid='review-title']",
                    ]:
                        try:
                            t = cards[0].find_element(By.CSS_SELECTOR, sel).text.strip()
                            if t:
                                first_text = t
                                break
                        except Exception:
                            pass
                    if first_text and first_text != snapshot_before:
                        return True
                except Exception:
                    pass

            print(f"   ⚠️  Contenu inchangé après 30s (tentative {attempt}/{MAX_RETRIES})")

            if attempt < MAX_RETRIES:
                print(f"   🔄 Nouvelle tentative dans 3s...")
                self.driver.execute_script("window.scrollTo(0, 0);")
                time.sleep(3)

        print("   ❌ Échec après 3 tentatives.")
        return False

    def _parse_card(self, card, hotel_name, hotel_city):
        name = self._first_text(
            card,
            "div.b08850ce41.f546354b44",
            "div[data-testid='reviewer-name'] span",
            "p.reviewer_name",
        )
        nationality = self._first_text(
            card,
            "span.d838fb5f41.aea5eccb71",
            "span[class*='reviewer_country']",
            "div[data-testid='reviewer-country'] span",
        )
        raw_rating = self._first_text(
            card,
            "div[data-testid='review-score'] div.bc946a29db",
            "div[data-testid='review-score']",
            "div.bui-review-score__badge",
        )
        rating    = self._extract_number(raw_rating)
        title     = self._first_text(card, "h4[data-testid='review-title']", "h3[class*='title']")
        pub_date  = self._first_text(card, "span[data-testid='review-date']", "p[data-testid='review-date']")
        pub_date  = re.sub(r'^Reviewed\s*:\s*', '', pub_date).strip()
        pub_date  = re.sub(r'^Rédigé le\s*:\s*', '', pub_date).strip()
        stay_date = self._first_text(card, "span[data-testid='review-stay-date']", "p.review_staydate")
        stay_date = re.sub(r'^Stayed\s*:\s*', '', stay_date).strip()
        stay_date = re.sub(r'^Séjour\s*:\s*', '', stay_date).strip()
        trip_type = self._first_text(
            card,
            "span[data-testid='review-traveler-type']",
            "p[data-testid='travel-type']",
            "div[data-testid='review-travel-type']",
        )
        pos  = self._text(card, "div[data-testid='review-positive-text']")
        neg  = self._text(card, "div[data-testid='review-negative-text']")
        full = (pos + " " + neg).strip()

        sub_scores = {}
        try:
            subs = card.find_elements(By.CSS_SELECTOR, "div[data-testid='review-subscore']")
            for s in subs:
                try:
                    cat = s.find_element(By.CSS_SELECTOR, "span").text.strip()
                    val = s.find_element(By.CSS_SELECTOR, "div.bc946a29db").text.strip()
                    sub_scores[cat] = self._extract_number(val)
                except Exception:
                    pass
        except Exception:
            pass

        return {
            "source": "booking.com",
            "hotel_name": hotel_name,
            "hotel_city": hotel_city,
            "reviewer_name": name,
            "nationality": nationality,
            "rating": rating,
            "review_title": title,
            "review_text": full,
            "positive_comment": pos,
            "negative_comment": neg,
            "publication_date": pub_date,
            "stay_date": stay_date,
            "trip_type": trip_type,
            "language": self.detect_language(full),
            **sub_scores,
        }

    def scrape_reviews(self, url, hotel_name, hotel_city, max_pages=90):
        print(f"🔄 Scraping : {hotel_name} ({hotel_city})")
        self.driver.get(url)
        time.sleep(6)

        all_reviews = []
        seen = set()
        page = 1

        while page <= max_pages:
            print(f"   📄 Page {page}  ({len(all_reviews)} avis cumulés)")

            try:
                self.wait.until(
                    EC.presence_of_element_located(
                        (By.CSS_SELECTOR, "div[data-testid='review-card']")
                    )
                )
            except Exception:
                print("   ⚠️  Aucune carte trouvée, fin du scraping.")
                break

            cards = self.driver.find_elements(
                By.CSS_SELECTOR, "div[data-testid='review-card']"
            )
            if not cards:
                print("   ⚠️  Liste de cartes vide, fin du scraping.")
                break

            snap = self._get_snapshot()

            new_count = 0
            for card in cards:
                review = self._parse_card(card, hotel_name, hotel_city)
                key = (
                    review.get("reviewer_name", ""),
                    review.get("publication_date", ""),
                    review.get("rating", ""),
                    review.get("review_text", "")[:50],
                )
                if key not in seen:
                    seen.add(key)
                    all_reviews.append(review)
                    new_count += 1

            print(f"      → {new_count} nouveaux avis")

            if new_count == 0:
                print("   ⚠️  Aucun nouvel avis — boucle détectée, arrêt.")
                break

            if not self._next_page(snap):
                print("   ✅ Dernière page atteinte.")
                break

            page += 1

        self.driver.quit()
        print(f"\n✅ {len(all_reviews)} avis extraits au total")
        return pd.DataFrame(all_reviews)


# ==================================================
if __name__ == "__main__":
    scraper = BookingScraper()
    df = scraper.scrape_reviews(URL, HOTEL_NAME, HOTEL_CITY, max_pages=MAX_PAGES)

    if not df.empty:
        filename = f"avis_{HOTEL_NAME.replace(' ', '_')}.csv"
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"\n💾 Fichier sauvegardé : {filename}  ({len(df)} avis)")

        print("\n🔍 Aperçu :")
        cols = [c for c in ["reviewer_name", "nationality", "rating",
                             "trip_type", "publication_date", "stay_date",
                             "language"] if c in df.columns]
        print(df[cols].head(10).to_string())

        print("\n📊 Colonnes vides :")
        empty = df[cols].isnull().sum() + (df[cols] == "").sum()
        print(empty[empty > 0] if empty[empty > 0].any() else "  Aucune colonne vide 🎉")
    else:
        print("❌ Aucun avis extrait.")

🔄 Scraping : Ibis Casablanca Voyageurs (Casablanca)
   📄 Page 1  (0 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 2  (10 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 3  (20 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 4  (30 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 5  (40 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 6  (50 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 7  (60 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tent

In [6]:
import time
import re
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from lingua import Language, LanguageDetectorBuilder

# ==================================================
URL = "https://www.booking.com/hotel/ma/ibis-casanearshore.html?aid=356980&label=gog235jc-10CAsojAFCJWliaXMtbW91c3NhZmlyLWNhc2FibGFuY2EtY2l0eS1jZW50ZXJIM1gDaIwBiAEBmAEzuAEHyAEM2AED6AEB-AEBiAIBqAIBuAKGhJ7PBsACAdICJGFlMzZiYjZmLWI3N2YtNGI1Zi1hMjljLWQ0MGIyMTg2NGIwNtgCAeACAQ&sid=53812c21f587e9070627ab5a462ac31d&dest_id=-28159&dest_type=city&dist=0&group_adults=2&group_children=0&hapos=13&hpos=13&nflt=ht_id%3D204%3Bclass%3D3&no_rooms=1&req_adults=2&req_children=0&room1=A%2CA&sb_price_type=total&sr_order=popularity&srepoch=1776785749&srpvid=9a3a6d8a9fed049f&type=total&ucfs=1&#tab-reviews"
HOTEL_NAME = "Ibis Casablanca Nearshore"
HOTEL_CITY = "Casablanca"
MAX_PAGES = 120
# ==================================================


class BookingScraper:
    def __init__(self):
        options = Options()
        options.add_argument('--lang=en')
        options.add_argument('--disable-blink-features=AutomationControlled')
        options.add_experimental_option("excludeSwitches", ["enable-automation"])
        options.add_experimental_option('useAutomationExtension', False)
        options.add_argument(
            '--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
            'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
        )
        self.driver = webdriver.Chrome(
            service=Service(ChromeDriverManager().install()), options=options
        )
        self.wait = WebDriverWait(self.driver, 20)

        self.detector = LanguageDetectorBuilder.from_languages(
            Language.FRENCH, Language.ENGLISH, Language.GERMAN,
            Language.SPANISH, Language.ITALIAN, Language.PORTUGUESE,
            Language.DUTCH, Language.RUSSIAN, Language.ARABIC, Language.CHINESE,
        ).with_minimum_relative_distance(0.15).build()

        self._lang_map = {
            Language.FRENCH: "french", Language.ENGLISH: "english",
            Language.GERMAN: "german", Language.SPANISH: "spanish",
            Language.ITALIAN: "italian", Language.PORTUGUESE: "portuguese",
            Language.DUTCH: "dutch", Language.RUSSIAN: "russian",
            Language.ARABIC: "arabic", Language.CHINESE: "chinese",
        }

    def _text(self, parent, css):
        try:
            return parent.find_element(By.CSS_SELECTOR, css).text.strip()
        except Exception:
            return ""

    def _first_text(self, parent, *selectors):
        for sel in selectors:
            val = self._text(parent, sel)
            if val:
                return val
        return ""

    def _extract_number(self, raw):
        if not raw:
            return ""
        raw = raw.replace(",", ".")
        match = re.search(r'\d+(?:\.\d+)?', raw)
        return match.group(0) if match else ""

    def detect_language(self, text):
        if not text or len(text.strip()) < 5:
            return "unknown"
        try:
            lang = self.detector.detect_language_of(text)
            return self._lang_map.get(lang, "unknown") if lang else "unknown"
        except Exception:
            return "unknown"

    def _get_snapshot(self):
        try:
            card = self.driver.find_element(By.CSS_SELECTOR, "div[data-testid='review-card']")
            for sel in [
                "div[data-testid='review-positive-text']",
                "div[data-testid='review-negative-text']",
                "h4[data-testid='review-title']",
            ]:
                try:
                    t = card.find_element(By.CSS_SELECTOR, sel).text.strip()
                    if t:
                        return t
                except Exception:
                    pass
        except Exception:
            pass
        return ""

    def _next_page(self, snapshot_before):
        MAX_RETRIES = 3

        for attempt in range(1, MAX_RETRIES + 1):
            btn = None

            labels = ["Page suivante", "Suivant", "Next page", "Next"]
            for label in labels:
                try:
                    btn = self.driver.find_element(
                        By.CSS_SELECTOR, f"button[aria-label='{label}']"
                    )
                    print(f"      🔘 Bouton trouvé : '{label}'")
                    break
                except Exception:
                    pass

            if btn is None:
                try:
                    btn = self.driver.find_element(
                        By.XPATH,
                        "//button[contains(@aria-label,'suivante') or "
                        "contains(@aria-label,'Suivant') or "
                        "contains(@aria-label,'Next')]"
                    )
                    print(f"      🔘 Bouton via XPath : '{btn.get_attribute('aria-label')}'")
                except Exception:
                    pass

            if btn is None:
                print("   ⚠️  Bouton introuvable.")
                return False

            disabled      = btn.get_attribute("disabled")
            aria_disabled = btn.get_attribute("aria-disabled") or ""
            classes       = btn.get_attribute("class") or ""
            if disabled or aria_disabled == "true" or "disabled" in classes:
                print("   ✅ Bouton désactivé — dernière page atteinte.")
                return False

            try:
                self.driver.execute_script(
                    "arguments[0].scrollIntoView({block:'center'});", btn
                )
                time.sleep(1)
            except Exception:
                pass

            clicked = False
            for method in ['normal', 'js']:
                try:
                    if method == 'normal':
                        btn.click()
                    else:
                        self.driver.execute_script("arguments[0].click();", btn)
                    clicked = True
                    print(f"      ✅ Clic OK ({method}) — tentative {attempt}/{MAX_RETRIES}")
                    break
                except Exception as e:
                    print(f"      ⚠️  Clic {method} échoué : {e}")

            if not clicked:
                time.sleep(2)
                continue

            for _ in range(60):
                time.sleep(0.5)
                try:
                    cards = self.driver.find_elements(
                        By.CSS_SELECTOR, "div[data-testid='review-card']"
                    )
                    if not cards:
                        continue
                    first_text = ""
                    for sel in [
                        "div[data-testid='review-positive-text']",
                        "div[data-testid='review-negative-text']",
                        "h4[data-testid='review-title']",
                    ]:
                        try:
                            t = cards[0].find_element(By.CSS_SELECTOR, sel).text.strip()
                            if t:
                                first_text = t
                                break
                        except Exception:
                            pass
                    if first_text and first_text != snapshot_before:
                        return True
                except Exception:
                    pass

            print(f"   ⚠️  Contenu inchangé après 30s (tentative {attempt}/{MAX_RETRIES})")

            if attempt < MAX_RETRIES:
                print(f"   🔄 Nouvelle tentative dans 3s...")
                self.driver.execute_script("window.scrollTo(0, 0);")
                time.sleep(3)

        print("   ❌ Échec après 3 tentatives.")
        return False

    def _parse_card(self, card, hotel_name, hotel_city):
        name = self._first_text(
            card,
            "div.b08850ce41.f546354b44",
            "div[data-testid='reviewer-name'] span",
            "p.reviewer_name",
        )
        nationality = self._first_text(
            card,
            "span.d838fb5f41.aea5eccb71",
            "span[class*='reviewer_country']",
            "div[data-testid='reviewer-country'] span",
        )
        raw_rating = self._first_text(
            card,
            "div[data-testid='review-score'] div.bc946a29db",
            "div[data-testid='review-score']",
            "div.bui-review-score__badge",
        )
        rating    = self._extract_number(raw_rating)
        title     = self._first_text(card, "h4[data-testid='review-title']", "h3[class*='title']")
        pub_date  = self._first_text(card, "span[data-testid='review-date']", "p[data-testid='review-date']")
        pub_date  = re.sub(r'^Reviewed\s*:\s*', '', pub_date).strip()
        pub_date  = re.sub(r'^Rédigé le\s*:\s*', '', pub_date).strip()
        stay_date = self._first_text(card, "span[data-testid='review-stay-date']", "p.review_staydate")
        stay_date = re.sub(r'^Stayed\s*:\s*', '', stay_date).strip()
        stay_date = re.sub(r'^Séjour\s*:\s*', '', stay_date).strip()
        trip_type = self._first_text(
            card,
            "span[data-testid='review-traveler-type']",
            "p[data-testid='travel-type']",
            "div[data-testid='review-travel-type']",
        )
        pos  = self._text(card, "div[data-testid='review-positive-text']")
        neg  = self._text(card, "div[data-testid='review-negative-text']")
        full = (pos + " " + neg).strip()

        sub_scores = {}
        try:
            subs = card.find_elements(By.CSS_SELECTOR, "div[data-testid='review-subscore']")
            for s in subs:
                try:
                    cat = s.find_element(By.CSS_SELECTOR, "span").text.strip()
                    val = s.find_element(By.CSS_SELECTOR, "div.bc946a29db").text.strip()
                    sub_scores[cat] = self._extract_number(val)
                except Exception:
                    pass
        except Exception:
            pass

        return {
            "source": "booking.com",
            "hotel_name": hotel_name,
            "hotel_city": hotel_city,
            "reviewer_name": name,
            "nationality": nationality,
            "rating": rating,
            "review_title": title,
            "review_text": full,
            "positive_comment": pos,
            "negative_comment": neg,
            "publication_date": pub_date,
            "stay_date": stay_date,
            "trip_type": trip_type,
            "language": self.detect_language(full),
            **sub_scores,
        }

    def scrape_reviews(self, url, hotel_name, hotel_city, max_pages=90):
        print(f"🔄 Scraping : {hotel_name} ({hotel_city})")
        self.driver.get(url)
        time.sleep(6)

        all_reviews = []
        seen = set()
        page = 1

        while page <= max_pages:
            print(f"   📄 Page {page}  ({len(all_reviews)} avis cumulés)")

            try:
                self.wait.until(
                    EC.presence_of_element_located(
                        (By.CSS_SELECTOR, "div[data-testid='review-card']")
                    )
                )
            except Exception:
                print("   ⚠️  Aucune carte trouvée, fin du scraping.")
                break

            cards = self.driver.find_elements(
                By.CSS_SELECTOR, "div[data-testid='review-card']"
            )
            if not cards:
                print("   ⚠️  Liste de cartes vide, fin du scraping.")
                break

            snap = self._get_snapshot()

            new_count = 0
            for card in cards:
                review = self._parse_card(card, hotel_name, hotel_city)
                key = (
                    review.get("reviewer_name", ""),
                    review.get("publication_date", ""),
                    review.get("rating", ""),
                    review.get("review_text", "")[:50],
                )
                if key not in seen:
                    seen.add(key)
                    all_reviews.append(review)
                    new_count += 1

            print(f"      → {new_count} nouveaux avis")

            if new_count == 0:
                print("   ⚠️  Aucun nouvel avis — boucle détectée, arrêt.")
                break

            if not self._next_page(snap):
                print("   ✅ Dernière page atteinte.")
                break

            page += 1

        self.driver.quit()
        print(f"\n✅ {len(all_reviews)} avis extraits au total")
        return pd.DataFrame(all_reviews)


# ==================================================
if __name__ == "__main__":
    scraper = BookingScraper()
    df = scraper.scrape_reviews(URL, HOTEL_NAME, HOTEL_CITY, max_pages=MAX_PAGES)

    if not df.empty:
        filename = f"avis_{HOTEL_NAME.replace(' ', '_')}.csv"
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"\n💾 Fichier sauvegardé : {filename}  ({len(df)} avis)")

        print("\n🔍 Aperçu :")
        cols = [c for c in ["reviewer_name", "nationality", "rating",
                             "trip_type", "publication_date", "stay_date",
                             "language"] if c in df.columns]
        print(df[cols].head(10).to_string())

        print("\n📊 Colonnes vides :")
        empty = df[cols].isnull().sum() + (df[cols] == "").sum()
        print(empty[empty > 0] if empty[empty > 0].any() else "  Aucune colonne vide 🎉")
    else:
        print("❌ Aucun avis extrait.")

🔄 Scraping : Ibis Casablanca Nearshore (Casablanca)
   📄 Page 1  (0 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 2  (10 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 3  (20 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 4  (30 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 5  (40 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 6  (50 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 7  (60 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tent

In [7]:
import time
import re
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from lingua import Language, LanguageDetectorBuilder

# ==================================================
URL = "https://www.booking.com/hotel/ma/ibis-moussafir-fes.fr.html?auth_success=1#tab-reviews"
HOTEL_NAME = "Ibis Moussafir Fès"
HOTEL_CITY = "Fès"
MAX_PAGES = 260
# ==================================================


class BookingScraper:
    def __init__(self):
        options = Options()
        options.add_argument('--lang=en')
        options.add_argument('--disable-blink-features=AutomationControlled')
        options.add_experimental_option("excludeSwitches", ["enable-automation"])
        options.add_experimental_option('useAutomationExtension', False)
        options.add_argument(
            '--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
            'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
        )
        self.driver = webdriver.Chrome(
            service=Service(ChromeDriverManager().install()), options=options
        )
        self.wait = WebDriverWait(self.driver, 20)

        self.detector = LanguageDetectorBuilder.from_languages(
            Language.FRENCH, Language.ENGLISH, Language.GERMAN,
            Language.SPANISH, Language.ITALIAN, Language.PORTUGUESE,
            Language.DUTCH, Language.RUSSIAN, Language.ARABIC, Language.CHINESE,
        ).with_minimum_relative_distance(0.15).build()

        self._lang_map = {
            Language.FRENCH: "french", Language.ENGLISH: "english",
            Language.GERMAN: "german", Language.SPANISH: "spanish",
            Language.ITALIAN: "italian", Language.PORTUGUESE: "portuguese",
            Language.DUTCH: "dutch", Language.RUSSIAN: "russian",
            Language.ARABIC: "arabic", Language.CHINESE: "chinese",
        }

    def _text(self, parent, css):
        try:
            return parent.find_element(By.CSS_SELECTOR, css).text.strip()
        except Exception:
            return ""

    def _first_text(self, parent, *selectors):
        for sel in selectors:
            val = self._text(parent, sel)
            if val:
                return val
        return ""

    def _extract_number(self, raw):
        if not raw:
            return ""
        raw = raw.replace(",", ".")
        match = re.search(r'\d+(?:\.\d+)?', raw)
        return match.group(0) if match else ""

    def detect_language(self, text):
        if not text or len(text.strip()) < 5:
            return "unknown"
        try:
            lang = self.detector.detect_language_of(text)
            return self._lang_map.get(lang, "unknown") if lang else "unknown"
        except Exception:
            return "unknown"

    def _get_snapshot(self):
        try:
            card = self.driver.find_element(By.CSS_SELECTOR, "div[data-testid='review-card']")
            for sel in [
                "div[data-testid='review-positive-text']",
                "div[data-testid='review-negative-text']",
                "h4[data-testid='review-title']",
            ]:
                try:
                    t = card.find_element(By.CSS_SELECTOR, sel).text.strip()
                    if t:
                        return t
                except Exception:
                    pass
        except Exception:
            pass
        return ""

    def _next_page(self, snapshot_before):
        MAX_RETRIES = 3

        for attempt in range(1, MAX_RETRIES + 1):
            btn = None

            labels = ["Page suivante", "Suivant", "Next page", "Next"]
            for label in labels:
                try:
                    btn = self.driver.find_element(
                        By.CSS_SELECTOR, f"button[aria-label='{label}']"
                    )
                    print(f"      🔘 Bouton trouvé : '{label}'")
                    break
                except Exception:
                    pass

            if btn is None:
                try:
                    btn = self.driver.find_element(
                        By.XPATH,
                        "//button[contains(@aria-label,'suivante') or "
                        "contains(@aria-label,'Suivant') or "
                        "contains(@aria-label,'Next')]"
                    )
                    print(f"      🔘 Bouton via XPath : '{btn.get_attribute('aria-label')}'")
                except Exception:
                    pass

            if btn is None:
                print("   ⚠️  Bouton introuvable.")
                return False

            disabled      = btn.get_attribute("disabled")
            aria_disabled = btn.get_attribute("aria-disabled") or ""
            classes       = btn.get_attribute("class") or ""
            if disabled or aria_disabled == "true" or "disabled" in classes:
                print("   ✅ Bouton désactivé — dernière page atteinte.")
                return False

            try:
                self.driver.execute_script(
                    "arguments[0].scrollIntoView({block:'center'});", btn
                )
                time.sleep(1)
            except Exception:
                pass

            clicked = False
            for method in ['normal', 'js']:
                try:
                    if method == 'normal':
                        btn.click()
                    else:
                        self.driver.execute_script("arguments[0].click();", btn)
                    clicked = True
                    print(f"      ✅ Clic OK ({method}) — tentative {attempt}/{MAX_RETRIES}")
                    break
                except Exception as e:
                    print(f"      ⚠️  Clic {method} échoué : {e}")

            if not clicked:
                time.sleep(2)
                continue

            for _ in range(60):
                time.sleep(0.5)
                try:
                    cards = self.driver.find_elements(
                        By.CSS_SELECTOR, "div[data-testid='review-card']"
                    )
                    if not cards:
                        continue
                    first_text = ""
                    for sel in [
                        "div[data-testid='review-positive-text']",
                        "div[data-testid='review-negative-text']",
                        "h4[data-testid='review-title']",
                    ]:
                        try:
                            t = cards[0].find_element(By.CSS_SELECTOR, sel).text.strip()
                            if t:
                                first_text = t
                                break
                        except Exception:
                            pass
                    if first_text and first_text != snapshot_before:
                        return True
                except Exception:
                    pass

            print(f"   ⚠️  Contenu inchangé après 30s (tentative {attempt}/{MAX_RETRIES})")

            if attempt < MAX_RETRIES:
                print(f"   🔄 Nouvelle tentative dans 3s...")
                self.driver.execute_script("window.scrollTo(0, 0);")
                time.sleep(3)

        print("   ❌ Échec après 3 tentatives.")
        return False

    def _parse_card(self, card, hotel_name, hotel_city):
        name = self._first_text(
            card,
            "div.b08850ce41.f546354b44",
            "div[data-testid='reviewer-name'] span",
            "p.reviewer_name",
        )
        nationality = self._first_text(
            card,
            "span.d838fb5f41.aea5eccb71",
            "span[class*='reviewer_country']",
            "div[data-testid='reviewer-country'] span",
        )
        raw_rating = self._first_text(
            card,
            "div[data-testid='review-score'] div.bc946a29db",
            "div[data-testid='review-score']",
            "div.bui-review-score__badge",
        )
        rating    = self._extract_number(raw_rating)
        title     = self._first_text(card, "h4[data-testid='review-title']", "h3[class*='title']")
        pub_date  = self._first_text(card, "span[data-testid='review-date']", "p[data-testid='review-date']")
        pub_date  = re.sub(r'^Reviewed\s*:\s*', '', pub_date).strip()
        pub_date  = re.sub(r'^Rédigé le\s*:\s*', '', pub_date).strip()
        stay_date = self._first_text(card, "span[data-testid='review-stay-date']", "p.review_staydate")
        stay_date = re.sub(r'^Stayed\s*:\s*', '', stay_date).strip()
        stay_date = re.sub(r'^Séjour\s*:\s*', '', stay_date).strip()
        trip_type = self._first_text(
            card,
            "span[data-testid='review-traveler-type']",
            "p[data-testid='travel-type']",
            "div[data-testid='review-travel-type']",
        )
        pos  = self._text(card, "div[data-testid='review-positive-text']")
        neg  = self._text(card, "div[data-testid='review-negative-text']")
        full = (pos + " " + neg).strip()

        sub_scores = {}
        try:
            subs = card.find_elements(By.CSS_SELECTOR, "div[data-testid='review-subscore']")
            for s in subs:
                try:
                    cat = s.find_element(By.CSS_SELECTOR, "span").text.strip()
                    val = s.find_element(By.CSS_SELECTOR, "div.bc946a29db").text.strip()
                    sub_scores[cat] = self._extract_number(val)
                except Exception:
                    pass
        except Exception:
            pass

        return {
            "source": "booking.com",
            "hotel_name": hotel_name,
            "hotel_city": hotel_city,
            "reviewer_name": name,
            "nationality": nationality,
            "rating": rating,
            "review_title": title,
            "review_text": full,
            "positive_comment": pos,
            "negative_comment": neg,
            "publication_date": pub_date,
            "stay_date": stay_date,
            "trip_type": trip_type,
            "language": self.detect_language(full),
            **sub_scores,
        }

    def scrape_reviews(self, url, hotel_name, hotel_city, max_pages=90):
        print(f"🔄 Scraping : {hotel_name} ({hotel_city})")
        self.driver.get(url)
        time.sleep(6)

        all_reviews = []
        seen = set()
        page = 1

        while page <= max_pages:
            print(f"   📄 Page {page}  ({len(all_reviews)} avis cumulés)")

            try:
                self.wait.until(
                    EC.presence_of_element_located(
                        (By.CSS_SELECTOR, "div[data-testid='review-card']")
                    )
                )
            except Exception:
                print("   ⚠️  Aucune carte trouvée, fin du scraping.")
                break

            cards = self.driver.find_elements(
                By.CSS_SELECTOR, "div[data-testid='review-card']"
            )
            if not cards:
                print("   ⚠️  Liste de cartes vide, fin du scraping.")
                break

            snap = self._get_snapshot()

            new_count = 0
            for card in cards:
                review = self._parse_card(card, hotel_name, hotel_city)
                key = (
                    review.get("reviewer_name", ""),
                    review.get("publication_date", ""),
                    review.get("rating", ""),
                    review.get("review_text", "")[:50],
                )
                if key not in seen:
                    seen.add(key)
                    all_reviews.append(review)
                    new_count += 1

            print(f"      → {new_count} nouveaux avis")

            if new_count == 0:
                print("   ⚠️  Aucun nouvel avis — boucle détectée, arrêt.")
                break

            if not self._next_page(snap):
                print("   ✅ Dernière page atteinte.")
                break

            page += 1

        self.driver.quit()
        print(f"\n✅ {len(all_reviews)} avis extraits au total")
        return pd.DataFrame(all_reviews)


# ==================================================
if __name__ == "__main__":
    scraper = BookingScraper()
    df = scraper.scrape_reviews(URL, HOTEL_NAME, HOTEL_CITY, max_pages=MAX_PAGES)

    if not df.empty:
        filename = f"avis_{HOTEL_NAME.replace(' ', '_')}.csv"
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"\n💾 Fichier sauvegardé : {filename}  ({len(df)} avis)")

        print("\n🔍 Aperçu :")
        cols = [c for c in ["reviewer_name", "nationality", "rating",
                             "trip_type", "publication_date", "stay_date",
                             "language"] if c in df.columns]
        print(df[cols].head(10).to_string())

        print("\n📊 Colonnes vides :")
        empty = df[cols].isnull().sum() + (df[cols] == "").sum()
        print(empty[empty > 0] if empty[empty > 0].any() else "  Aucune colonne vide 🎉")
    else:
        print("❌ Aucun avis extrait.")

🔄 Scraping : Ibis Moussafir Fès (Fès)
   📄 Page 1  (0 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 2  (10 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 3  (20 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 4  (30 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 5  (40 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 6  (50 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 7  (60 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄

In [8]:
import time
import re
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from lingua import Language, LanguageDetectorBuilder

# ==================================================
URL = "https://www.booking.com/hotel/ma/ibis-rabat-agdal.html?aid=356980&label=gog235jc-10CAsojAFCEGliaXMtcmFiYXQtYWdkYWxIM1gDaIwBiAEBmAEzuAEHyAEM2AED6AEB-AEBiAIBqAIBuALV2aPPBsACAdICJDg3YzJkNWJhLTQwYjQtNDhjMi1hM2YyLTg5YTdiNWJiMzJhONgCAeACAQ&sid=53812c21f587e9070627ab5a462ac31d&dest_id=-43376&dest_type=city&dist=0&group_adults=2&group_children=0&hapos=1&hpos=1&no_rooms=1&req_adults=2&req_children=0&room1=A%2CA&sb_price_type=total&sr_order=popularity&srepoch=1776872670&srpvid=a1986eaba28a035f&type=total&ucfs=1&#tab-reviews"
HOTEL_NAME = "Ibis Rabat Agdal"
HOTEL_CITY = "Rabat"
MAX_PAGES = 200
# ==================================================


class BookingScraper:
    def __init__(self):
        options = Options()
        options.add_argument('--lang=en')
        options.add_argument('--disable-blink-features=AutomationControlled')
        options.add_experimental_option("excludeSwitches", ["enable-automation"])
        options.add_experimental_option('useAutomationExtension', False)
        options.add_argument(
            '--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
            'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
        )
        self.driver = webdriver.Chrome(
            service=Service(ChromeDriverManager().install()), options=options
        )
        self.wait = WebDriverWait(self.driver, 20)

        self.detector = LanguageDetectorBuilder.from_languages(
            Language.FRENCH, Language.ENGLISH, Language.GERMAN,
            Language.SPANISH, Language.ITALIAN, Language.PORTUGUESE,
            Language.DUTCH, Language.RUSSIAN, Language.ARABIC, Language.CHINESE,
        ).with_minimum_relative_distance(0.15).build()

        self._lang_map = {
            Language.FRENCH: "french", Language.ENGLISH: "english",
            Language.GERMAN: "german", Language.SPANISH: "spanish",
            Language.ITALIAN: "italian", Language.PORTUGUESE: "portuguese",
            Language.DUTCH: "dutch", Language.RUSSIAN: "russian",
            Language.ARABIC: "arabic", Language.CHINESE: "chinese",
        }

    def _text(self, parent, css):
        try:
            return parent.find_element(By.CSS_SELECTOR, css).text.strip()
        except Exception:
            return ""

    def _first_text(self, parent, *selectors):
        for sel in selectors:
            val = self._text(parent, sel)
            if val:
                return val
        return ""

    def _extract_number(self, raw):
        if not raw:
            return ""
        raw = raw.replace(",", ".")
        match = re.search(r'\d+(?:\.\d+)?', raw)
        return match.group(0) if match else ""

    def detect_language(self, text):
        if not text or len(text.strip()) < 5:
            return "unknown"
        try:
            lang = self.detector.detect_language_of(text)
            return self._lang_map.get(lang, "unknown") if lang else "unknown"
        except Exception:
            return "unknown"

    def _get_snapshot(self):
        try:
            card = self.driver.find_element(By.CSS_SELECTOR, "div[data-testid='review-card']")
            for sel in [
                "div[data-testid='review-positive-text']",
                "div[data-testid='review-negative-text']",
                "h4[data-testid='review-title']",
            ]:
                try:
                    t = card.find_element(By.CSS_SELECTOR, sel).text.strip()
                    if t:
                        return t
                except Exception:
                    pass
        except Exception:
            pass
        return ""

    def _next_page(self, snapshot_before):
        MAX_RETRIES = 3

        for attempt in range(1, MAX_RETRIES + 1):
            btn = None

            labels = ["Page suivante", "Suivant", "Next page", "Next"]
            for label in labels:
                try:
                    btn = self.driver.find_element(
                        By.CSS_SELECTOR, f"button[aria-label='{label}']"
                    )
                    print(f"      🔘 Bouton trouvé : '{label}'")
                    break
                except Exception:
                    pass

            if btn is None:
                try:
                    btn = self.driver.find_element(
                        By.XPATH,
                        "//button[contains(@aria-label,'suivante') or "
                        "contains(@aria-label,'Suivant') or "
                        "contains(@aria-label,'Next')]"
                    )
                    print(f"      🔘 Bouton via XPath : '{btn.get_attribute('aria-label')}'")
                except Exception:
                    pass

            if btn is None:
                print("   ⚠️  Bouton introuvable.")
                return False

            disabled      = btn.get_attribute("disabled")
            aria_disabled = btn.get_attribute("aria-disabled") or ""
            classes       = btn.get_attribute("class") or ""
            if disabled or aria_disabled == "true" or "disabled" in classes:
                print("   ✅ Bouton désactivé — dernière page atteinte.")
                return False

            try:
                self.driver.execute_script(
                    "arguments[0].scrollIntoView({block:'center'});", btn
                )
                time.sleep(1)
            except Exception:
                pass

            clicked = False
            for method in ['normal', 'js']:
                try:
                    if method == 'normal':
                        btn.click()
                    else:
                        self.driver.execute_script("arguments[0].click();", btn)
                    clicked = True
                    print(f"      ✅ Clic OK ({method}) — tentative {attempt}/{MAX_RETRIES}")
                    break
                except Exception as e:
                    print(f"      ⚠️  Clic {method} échoué : {e}")

            if not clicked:
                time.sleep(2)
                continue

            for _ in range(60):
                time.sleep(0.5)
                try:
                    cards = self.driver.find_elements(
                        By.CSS_SELECTOR, "div[data-testid='review-card']"
                    )
                    if not cards:
                        continue
                    first_text = ""
                    for sel in [
                        "div[data-testid='review-positive-text']",
                        "div[data-testid='review-negative-text']",
                        "h4[data-testid='review-title']",
                    ]:
                        try:
                            t = cards[0].find_element(By.CSS_SELECTOR, sel).text.strip()
                            if t:
                                first_text = t
                                break
                        except Exception:
                            pass
                    if first_text and first_text != snapshot_before:
                        return True
                except Exception:
                    pass

            print(f"   ⚠️  Contenu inchangé après 30s (tentative {attempt}/{MAX_RETRIES})")

            if attempt < MAX_RETRIES:
                print(f"   🔄 Nouvelle tentative dans 3s...")
                self.driver.execute_script("window.scrollTo(0, 0);")
                time.sleep(3)

        print("   ❌ Échec après 3 tentatives.")
        return False

    def _parse_card(self, card, hotel_name, hotel_city):
        name = self._first_text(
            card,
            "div.b08850ce41.f546354b44",
            "div[data-testid='reviewer-name'] span",
            "p.reviewer_name",
        )
        nationality = self._first_text(
            card,
            "span.d838fb5f41.aea5eccb71",
            "span[class*='reviewer_country']",
            "div[data-testid='reviewer-country'] span",
        )
        raw_rating = self._first_text(
            card,
            "div[data-testid='review-score'] div.bc946a29db",
            "div[data-testid='review-score']",
            "div.bui-review-score__badge",
        )
        rating    = self._extract_number(raw_rating)
        title     = self._first_text(card, "h4[data-testid='review-title']", "h3[class*='title']")
        pub_date  = self._first_text(card, "span[data-testid='review-date']", "p[data-testid='review-date']")
        pub_date  = re.sub(r'^Reviewed\s*:\s*', '', pub_date).strip()
        pub_date  = re.sub(r'^Rédigé le\s*:\s*', '', pub_date).strip()
        stay_date = self._first_text(card, "span[data-testid='review-stay-date']", "p.review_staydate")
        stay_date = re.sub(r'^Stayed\s*:\s*', '', stay_date).strip()
        stay_date = re.sub(r'^Séjour\s*:\s*', '', stay_date).strip()
        trip_type = self._first_text(
            card,
            "span[data-testid='review-traveler-type']",
            "p[data-testid='travel-type']",
            "div[data-testid='review-travel-type']",
        )
        pos  = self._text(card, "div[data-testid='review-positive-text']")
        neg  = self._text(card, "div[data-testid='review-negative-text']")
        full = (pos + " " + neg).strip()

        sub_scores = {}
        try:
            subs = card.find_elements(By.CSS_SELECTOR, "div[data-testid='review-subscore']")
            for s in subs:
                try:
                    cat = s.find_element(By.CSS_SELECTOR, "span").text.strip()
                    val = s.find_element(By.CSS_SELECTOR, "div.bc946a29db").text.strip()
                    sub_scores[cat] = self._extract_number(val)
                except Exception:
                    pass
        except Exception:
            pass

        return {
            "source": "booking.com",
            "hotel_name": hotel_name,
            "hotel_city": hotel_city,
            "reviewer_name": name,
            "nationality": nationality,
            "rating": rating,
            "review_title": title,
            "review_text": full,
            "positive_comment": pos,
            "negative_comment": neg,
            "publication_date": pub_date,
            "stay_date": stay_date,
            "trip_type": trip_type,
            "language": self.detect_language(full),
            **sub_scores,
        }

    def scrape_reviews(self, url, hotel_name, hotel_city, max_pages=90):
        print(f"🔄 Scraping : {hotel_name} ({hotel_city})")
        self.driver.get(url)
        time.sleep(6)

        all_reviews = []
        seen = set()
        page = 1

        while page <= max_pages:
            print(f"   📄 Page {page}  ({len(all_reviews)} avis cumulés)")

            try:
                self.wait.until(
                    EC.presence_of_element_located(
                        (By.CSS_SELECTOR, "div[data-testid='review-card']")
                    )
                )
            except Exception:
                print("   ⚠️  Aucune carte trouvée, fin du scraping.")
                break

            cards = self.driver.find_elements(
                By.CSS_SELECTOR, "div[data-testid='review-card']"
            )
            if not cards:
                print("   ⚠️  Liste de cartes vide, fin du scraping.")
                break

            snap = self._get_snapshot()

            new_count = 0
            for card in cards:
                review = self._parse_card(card, hotel_name, hotel_city)
                key = (
                    review.get("reviewer_name", ""),
                    review.get("publication_date", ""),
                    review.get("rating", ""),
                    review.get("review_text", "")[:50],
                )
                if key not in seen:
                    seen.add(key)
                    all_reviews.append(review)
                    new_count += 1

            print(f"      → {new_count} nouveaux avis")

            if new_count == 0:
                print("   ⚠️  Aucun nouvel avis — boucle détectée, arrêt.")
                break

            if not self._next_page(snap):
                print("   ✅ Dernière page atteinte.")
                break

            page += 1

        self.driver.quit()
        print(f"\n✅ {len(all_reviews)} avis extraits au total")
        return pd.DataFrame(all_reviews)


# ==================================================
if __name__ == "__main__":
    scraper = BookingScraper()
    df = scraper.scrape_reviews(URL, HOTEL_NAME, HOTEL_CITY, max_pages=MAX_PAGES)

    if not df.empty:
        filename = f"avis_{HOTEL_NAME.replace(' ', '_')}.csv"
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"\n💾 Fichier sauvegardé : {filename}  ({len(df)} avis)")

        print("\n🔍 Aperçu :")
        cols = [c for c in ["reviewer_name", "nationality", "rating",
                             "trip_type", "publication_date", "stay_date",
                             "language"] if c in df.columns]
        print(df[cols].head(10).to_string())

        print("\n📊 Colonnes vides :")
        empty = df[cols].isnull().sum() + (df[cols] == "").sum()
        print(empty[empty > 0] if empty[empty > 0].any() else "  Aucune colonne vide 🎉")
    else:
        print("❌ Aucun avis extrait.")

🔄 Scraping : Ibis Rabat Agdal (Rabat)
   📄 Page 1  (0 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 2  (10 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 3  (20 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 4  (30 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 5  (40 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 6  (50 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 7  (60 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄

In [6]:
df.head()

,source,hotel_name,hotel_city,reviewer_name,nationality,rating,review_title,review_text,positive_comment,negative_comment,publication_date,stay_date,trip_type,language
0,booking.com,Ibis Moussafir Fès,Fès,Marianne,Canada,6.0,"Séjour correcte, nous recommandons lors de tem...",L’emplacement et la terrasse Aucune climatisat...,L’emplacement et la terrasse,Aucune climatisation (froid) dans les chambres...,Commentaire envoyé le 20 avril 2026,avril 2026,Couple,french
1,booking.com,Ibis Moussafir Fès,Fès,Nezha,France,7.0,agreable,le personnel de la restauration au top très pr...,le personnel de la restauration au top très pr...,petit déjeuner pas assez varier .on aurait aim...,Commentaire envoyé le 20 avril 2026,mars 2026,Couple,french
2,booking.com,Ibis Moussafir Fès,Fès,Mesoudi,Maroc,6.0,Séjour a ibis fes,J'ai aimer l'emplacement et les gens d'acceuil...,J'ai aimer l'emplacement et les gens d'acceuil,Il faut changer les serviettes de la salle de ...,Commentaire envoyé le 19 avril 2026,avril 2026,Voyageur individuel,french
3,booking.com,Ibis Moussafir Fès,Fès,Sophie,Suisse,7.0,Bien,Jardin Bruit du train et rue côté gare,Jardin,Bruit du train et rue côté gare,Commentaire envoyé le 17 avril 2026,avril 2026,Voyageur individuel,french
4,booking.com,Ibis Moussafir Fès,Fès,Taoufiq,France,7.0,Quelques inconvénients bruit dans le couloir,Bien situé Odeur de la cigarette dans la chambre,Bien situé,Odeur de la cigarette dans la chambre,Commentaire envoyé le 13 avril 2026,avril 2026,Couple,french


In [9]:
import time
import re
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from lingua import Language, LanguageDetectorBuilder

# ==================================================
URL = "https://www.booking.com/hotel/ma/ibis-moussafir-tanger-city-center.html?aid=356980&label=gog235jc-10CAsojAFCIWliaXMtbW91c3NhZmlyLXRhbmdlci1jaXR5LWNlbnRlckgzWANojAGIAQGYATO4AQfIAQ3YAQPoAQH4AQGIAgGoAgG4AsDgo88GwAIB0gIkZTJjYmZmYzQtZWVlOC00MTQ2LWI5MjYtYzEyMzViMDI1OTMy2AIB4AIB&sid=4ba5cffa630e66002a1045ea6b678064&dist=0&keep_landing=1&sb_price_type=total&type=total&lang=en-us&soz=1&lang_changed=1#tab-reviews"
HOTEL_NAME = "Ibis Tanger City Center"
HOTEL_CITY = "Tanger"
MAX_PAGES = 300
# ==================================================


class BookingScraper:
    def __init__(self):
        options = Options()
        options.add_argument('--lang=en')
        options.add_argument('--disable-blink-features=AutomationControlled')
        options.add_experimental_option("excludeSwitches", ["enable-automation"])
        options.add_experimental_option('useAutomationExtension', False)
        options.add_argument(
            '--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
            'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
        )
        self.driver = webdriver.Chrome(
            service=Service(ChromeDriverManager().install()), options=options
        )
        self.wait = WebDriverWait(self.driver, 20)

        self.detector = LanguageDetectorBuilder.from_languages(
            Language.FRENCH, Language.ENGLISH, Language.GERMAN,
            Language.SPANISH, Language.ITALIAN, Language.PORTUGUESE,
            Language.DUTCH, Language.RUSSIAN, Language.ARABIC, Language.CHINESE,
        ).with_minimum_relative_distance(0.15).build()

        self._lang_map = {
            Language.FRENCH: "french", Language.ENGLISH: "english",
            Language.GERMAN: "german", Language.SPANISH: "spanish",
            Language.ITALIAN: "italian", Language.PORTUGUESE: "portuguese",
            Language.DUTCH: "dutch", Language.RUSSIAN: "russian",
            Language.ARABIC: "arabic", Language.CHINESE: "chinese",
        }

    def _text(self, parent, css):
        try:
            return parent.find_element(By.CSS_SELECTOR, css).text.strip()
        except Exception:
            return ""

    def _first_text(self, parent, *selectors):
        for sel in selectors:
            val = self._text(parent, sel)
            if val:
                return val
        return ""

    def _extract_number(self, raw):
        if not raw:
            return ""
        raw = raw.replace(",", ".")
        match = re.search(r'\d+(?:\.\d+)?', raw)
        return match.group(0) if match else ""

    def detect_language(self, text):
        if not text or len(text.strip()) < 5:
            return "unknown"
        try:
            lang = self.detector.detect_language_of(text)
            return self._lang_map.get(lang, "unknown") if lang else "unknown"
        except Exception:
            return "unknown"

    def _get_snapshot(self):
        try:
            card = self.driver.find_element(By.CSS_SELECTOR, "div[data-testid='review-card']")
            for sel in [
                "div[data-testid='review-positive-text']",
                "div[data-testid='review-negative-text']",
                "h4[data-testid='review-title']",
            ]:
                try:
                    t = card.find_element(By.CSS_SELECTOR, sel).text.strip()
                    if t:
                        return t
                except Exception:
                    pass
        except Exception:
            pass
        return ""

    def _next_page(self, snapshot_before):
        MAX_RETRIES = 3

        for attempt in range(1, MAX_RETRIES + 1):
            btn = None

            labels = ["Page suivante", "Suivant", "Next page", "Next"]
            for label in labels:
                try:
                    btn = self.driver.find_element(
                        By.CSS_SELECTOR, f"button[aria-label='{label}']"
                    )
                    print(f"      🔘 Bouton trouvé : '{label}'")
                    break
                except Exception:
                    pass

            if btn is None:
                try:
                    btn = self.driver.find_element(
                        By.XPATH,
                        "//button[contains(@aria-label,'suivante') or "
                        "contains(@aria-label,'Suivant') or "
                        "contains(@aria-label,'Next')]"
                    )
                    print(f"      🔘 Bouton via XPath : '{btn.get_attribute('aria-label')}'")
                except Exception:
                    pass

            if btn is None:
                print("   ⚠️  Bouton introuvable.")
                return False

            disabled      = btn.get_attribute("disabled")
            aria_disabled = btn.get_attribute("aria-disabled") or ""
            classes       = btn.get_attribute("class") or ""
            if disabled or aria_disabled == "true" or "disabled" in classes:
                print("   ✅ Bouton désactivé — dernière page atteinte.")
                return False

            try:
                self.driver.execute_script(
                    "arguments[0].scrollIntoView({block:'center'});", btn
                )
                time.sleep(1)
            except Exception:
                pass

            clicked = False
            for method in ['normal', 'js']:
                try:
                    if method == 'normal':
                        btn.click()
                    else:
                        self.driver.execute_script("arguments[0].click();", btn)
                    clicked = True
                    print(f"      ✅ Clic OK ({method}) — tentative {attempt}/{MAX_RETRIES}")
                    break
                except Exception as e:
                    print(f"      ⚠️  Clic {method} échoué : {e}")

            if not clicked:
                time.sleep(2)
                continue

            for _ in range(60):
                time.sleep(0.5)
                try:
                    cards = self.driver.find_elements(
                        By.CSS_SELECTOR, "div[data-testid='review-card']"
                    )
                    if not cards:
                        continue
                    first_text = ""
                    for sel in [
                        "div[data-testid='review-positive-text']",
                        "div[data-testid='review-negative-text']",
                        "h4[data-testid='review-title']",
                    ]:
                        try:
                            t = cards[0].find_element(By.CSS_SELECTOR, sel).text.strip()
                            if t:
                                first_text = t
                                break
                        except Exception:
                            pass
                    if first_text and first_text != snapshot_before:
                        return True
                except Exception:
                    pass

            print(f"   ⚠️  Contenu inchangé après 30s (tentative {attempt}/{MAX_RETRIES})")

            if attempt < MAX_RETRIES:
                print(f"   🔄 Nouvelle tentative dans 3s...")
                self.driver.execute_script("window.scrollTo(0, 0);")
                time.sleep(3)

        print("   ❌ Échec après 3 tentatives.")
        return False

    def _parse_card(self, card, hotel_name, hotel_city):
        name = self._first_text(
            card,
            "div.b08850ce41.f546354b44",
            "div[data-testid='reviewer-name'] span",
            "p.reviewer_name",
        )
        nationality = self._first_text(
            card,
            "span.d838fb5f41.aea5eccb71",
            "span[class*='reviewer_country']",
            "div[data-testid='reviewer-country'] span",
        )
        raw_rating = self._first_text(
            card,
            "div[data-testid='review-score'] div.bc946a29db",
            "div[data-testid='review-score']",
            "div.bui-review-score__badge",
        )
        rating    = self._extract_number(raw_rating)
        title     = self._first_text(card, "h4[data-testid='review-title']", "h3[class*='title']")
        pub_date  = self._first_text(card, "span[data-testid='review-date']", "p[data-testid='review-date']")
        pub_date  = re.sub(r'^Reviewed\s*:\s*', '', pub_date).strip()
        pub_date  = re.sub(r'^Rédigé le\s*:\s*', '', pub_date).strip()
        stay_date = self._first_text(card, "span[data-testid='review-stay-date']", "p.review_staydate")
        stay_date = re.sub(r'^Stayed\s*:\s*', '', stay_date).strip()
        stay_date = re.sub(r'^Séjour\s*:\s*', '', stay_date).strip()
        trip_type = self._first_text(
            card,
            "span[data-testid='review-traveler-type']",
            "p[data-testid='travel-type']",
            "div[data-testid='review-travel-type']",
        )
        pos  = self._text(card, "div[data-testid='review-positive-text']")
        neg  = self._text(card, "div[data-testid='review-negative-text']")
        full = (pos + " " + neg).strip()

        sub_scores = {}
        try:
            subs = card.find_elements(By.CSS_SELECTOR, "div[data-testid='review-subscore']")
            for s in subs:
                try:
                    cat = s.find_element(By.CSS_SELECTOR, "span").text.strip()
                    val = s.find_element(By.CSS_SELECTOR, "div.bc946a29db").text.strip()
                    sub_scores[cat] = self._extract_number(val)
                except Exception:
                    pass
        except Exception:
            pass

        return {
            "source": "booking.com",
            "hotel_name": hotel_name,
            "hotel_city": hotel_city,
            "reviewer_name": name,
            "nationality": nationality,
            "rating": rating,
            "review_title": title,
            "review_text": full,
            "positive_comment": pos,
            "negative_comment": neg,
            "publication_date": pub_date,
            "stay_date": stay_date,
            "trip_type": trip_type,
            "language": self.detect_language(full),
            **sub_scores,
        }

    def scrape_reviews(self, url, hotel_name, hotel_city, max_pages=90):
        print(f"🔄 Scraping : {hotel_name} ({hotel_city})")
        self.driver.get(url)
        time.sleep(6)

        all_reviews = []
        seen = set()
        page = 1

        while page <= max_pages:
            print(f"   📄 Page {page}  ({len(all_reviews)} avis cumulés)")

            try:
                self.wait.until(
                    EC.presence_of_element_located(
                        (By.CSS_SELECTOR, "div[data-testid='review-card']")
                    )
                )
            except Exception:
                print("   ⚠️  Aucune carte trouvée, fin du scraping.")
                break

            cards = self.driver.find_elements(
                By.CSS_SELECTOR, "div[data-testid='review-card']"
            )
            if not cards:
                print("   ⚠️  Liste de cartes vide, fin du scraping.")
                break

            snap = self._get_snapshot()

            new_count = 0
            for card in cards:
                review = self._parse_card(card, hotel_name, hotel_city)
                key = (
                    review.get("reviewer_name", ""),
                    review.get("publication_date", ""),
                    review.get("rating", ""),
                    review.get("review_text", "")[:50],
                )
                if key not in seen:
                    seen.add(key)
                    all_reviews.append(review)
                    new_count += 1

            print(f"      → {new_count} nouveaux avis")

            if new_count == 0:
                print("   ⚠️  Aucun nouvel avis — boucle détectée, arrêt.")
                break

            if not self._next_page(snap):
                print("   ✅ Dernière page atteinte.")
                break

            page += 1

        self.driver.quit()
        print(f"\n✅ {len(all_reviews)} avis extraits au total")
        return pd.DataFrame(all_reviews)


# ==================================================
if __name__ == "__main__":
    scraper = BookingScraper()
    df = scraper.scrape_reviews(URL, HOTEL_NAME, HOTEL_CITY, max_pages=MAX_PAGES)

    if not df.empty:
        filename = f"avis_{HOTEL_NAME.replace(' ', '_')}.csv"
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"\n💾 Fichier sauvegardé : {filename}  ({len(df)} avis)")

        print("\n🔍 Aperçu :")
        cols = [c for c in ["reviewer_name", "nationality", "rating",
                             "trip_type", "publication_date", "stay_date",
                             "language"] if c in df.columns]
        print(df[cols].head(10).to_string())

        print("\n📊 Colonnes vides :")
        empty = df[cols].isnull().sum() + (df[cols] == "").sum()
        print(empty[empty > 0] if empty[empty > 0].any() else "  Aucune colonne vide 🎉")
    else:
        print("❌ Aucun avis extrait.")

🔄 Scraping : Ibis Tanger City Center (Tanger)
   📄 Page 1  (0 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Next page'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 2  (10 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Next page'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 3  (20 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Next page'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 4  (30 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Next page'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 5  (40 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Next page'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 6  (50 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Next page'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 7  (60 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Next page'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 8  (70 avis cu

In [10]:
import time
import re
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from lingua import Language, LanguageDetectorBuilder

# ==================================================
URL = "https://www.booking.com/hotel/ma/ibis-moussafir-marrakech-centre-gare.html?aid=356980&label=gog235jc-10CAsojAFCJGliaXMtbW91c3NhZmlyLW1hcnJha2VjaC1jZW50cmUtZ2FyZUgNWANojAGIAQGYATO4AQfIAQ3YAQPoAQH4AQGIAgGoAgG4Atnmo88GwAIB0gIkZmQwYjBjNDAtMTAxNC00MzQ0LTlhZDAtMjU1OThlYjAyNGUx2AIB4AIB&sid=4ba5cffa630e66002a1045ea6b678064&dist=0&keep_landing=1&sb_price_type=total&type=total&#tab-reviews"
HOTEL_NAME = "Ibis Marrakech Centre Gare"
HOTEL_CITY = "Marrakech"
MAX_PAGES = 250
# ==================================================


class BookingScraper:
    def __init__(self):
        options = Options()
        options.add_argument('--lang=en')
        options.add_argument('--disable-blink-features=AutomationControlled')
        options.add_experimental_option("excludeSwitches", ["enable-automation"])
        options.add_experimental_option('useAutomationExtension', False)
        options.add_argument(
            '--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
            'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
        )
        self.driver = webdriver.Chrome(
            service=Service(ChromeDriverManager().install()), options=options
        )
        self.wait = WebDriverWait(self.driver, 20)

        self.detector = LanguageDetectorBuilder.from_languages(
            Language.FRENCH, Language.ENGLISH, Language.GERMAN,
            Language.SPANISH, Language.ITALIAN, Language.PORTUGUESE,
            Language.DUTCH, Language.RUSSIAN, Language.ARABIC, Language.CHINESE,
        ).with_minimum_relative_distance(0.15).build()

        self._lang_map = {
            Language.FRENCH: "french", Language.ENGLISH: "english",
            Language.GERMAN: "german", Language.SPANISH: "spanish",
            Language.ITALIAN: "italian", Language.PORTUGUESE: "portuguese",
            Language.DUTCH: "dutch", Language.RUSSIAN: "russian",
            Language.ARABIC: "arabic", Language.CHINESE: "chinese",
        }

    def _text(self, parent, css):
        try:
            return parent.find_element(By.CSS_SELECTOR, css).text.strip()
        except Exception:
            return ""

    def _first_text(self, parent, *selectors):
        for sel in selectors:
            val = self._text(parent, sel)
            if val:
                return val
        return ""

    def _extract_number(self, raw):
        if not raw:
            return ""
        raw = raw.replace(",", ".")
        match = re.search(r'\d+(?:\.\d+)?', raw)
        return match.group(0) if match else ""

    def detect_language(self, text):
        if not text or len(text.strip()) < 5:
            return "unknown"
        try:
            lang = self.detector.detect_language_of(text)
            return self._lang_map.get(lang, "unknown") if lang else "unknown"
        except Exception:
            return "unknown"

    def _get_snapshot(self):
        try:
            card = self.driver.find_element(By.CSS_SELECTOR, "div[data-testid='review-card']")
            for sel in [
                "div[data-testid='review-positive-text']",
                "div[data-testid='review-negative-text']",
                "h4[data-testid='review-title']",
            ]:
                try:
                    t = card.find_element(By.CSS_SELECTOR, sel).text.strip()
                    if t:
                        return t
                except Exception:
                    pass
        except Exception:
            pass
        return ""

    def _next_page(self, snapshot_before):
        MAX_RETRIES = 3

        for attempt in range(1, MAX_RETRIES + 1):
            btn = None

            labels = ["Page suivante", "Suivant", "Next page", "Next"]
            for label in labels:
                try:
                    btn = self.driver.find_element(
                        By.CSS_SELECTOR, f"button[aria-label='{label}']"
                    )
                    print(f"      🔘 Bouton trouvé : '{label}'")
                    break
                except Exception:
                    pass

            if btn is None:
                try:
                    btn = self.driver.find_element(
                        By.XPATH,
                        "//button[contains(@aria-label,'suivante') or "
                        "contains(@aria-label,'Suivant') or "
                        "contains(@aria-label,'Next')]"
                    )
                    print(f"      🔘 Bouton via XPath : '{btn.get_attribute('aria-label')}'")
                except Exception:
                    pass

            if btn is None:
                print("   ⚠️  Bouton introuvable.")
                return False

            disabled      = btn.get_attribute("disabled")
            aria_disabled = btn.get_attribute("aria-disabled") or ""
            classes       = btn.get_attribute("class") or ""
            if disabled or aria_disabled == "true" or "disabled" in classes:
                print("   ✅ Bouton désactivé — dernière page atteinte.")
                return False

            try:
                self.driver.execute_script(
                    "arguments[0].scrollIntoView({block:'center'});", btn
                )
                time.sleep(1)
            except Exception:
                pass

            clicked = False
            for method in ['normal', 'js']:
                try:
                    if method == 'normal':
                        btn.click()
                    else:
                        self.driver.execute_script("arguments[0].click();", btn)
                    clicked = True
                    print(f"      ✅ Clic OK ({method}) — tentative {attempt}/{MAX_RETRIES}")
                    break
                except Exception as e:
                    print(f"      ⚠️  Clic {method} échoué : {e}")

            if not clicked:
                time.sleep(2)
                continue

            for _ in range(60):
                time.sleep(0.5)
                try:
                    cards = self.driver.find_elements(
                        By.CSS_SELECTOR, "div[data-testid='review-card']"
                    )
                    if not cards:
                        continue
                    first_text = ""
                    for sel in [
                        "div[data-testid='review-positive-text']",
                        "div[data-testid='review-negative-text']",
                        "h4[data-testid='review-title']",
                    ]:
                        try:
                            t = cards[0].find_element(By.CSS_SELECTOR, sel).text.strip()
                            if t:
                                first_text = t
                                break
                        except Exception:
                            pass
                    if first_text and first_text != snapshot_before:
                        return True
                except Exception:
                    pass

            print(f"   ⚠️  Contenu inchangé après 30s (tentative {attempt}/{MAX_RETRIES})")

            if attempt < MAX_RETRIES:
                print(f"   🔄 Nouvelle tentative dans 3s...")
                self.driver.execute_script("window.scrollTo(0, 0);")
                time.sleep(3)

        print("   ❌ Échec après 3 tentatives.")
        return False

    def _parse_card(self, card, hotel_name, hotel_city):
        name = self._first_text(
            card,
            "div.b08850ce41.f546354b44",
            "div[data-testid='reviewer-name'] span",
            "p.reviewer_name",
        )
        nationality = self._first_text(
            card,
            "span.d838fb5f41.aea5eccb71",
            "span[class*='reviewer_country']",
            "div[data-testid='reviewer-country'] span",
        )
        raw_rating = self._first_text(
            card,
            "div[data-testid='review-score'] div.bc946a29db",
            "div[data-testid='review-score']",
            "div.bui-review-score__badge",
        )
        rating    = self._extract_number(raw_rating)
        title     = self._first_text(card, "h4[data-testid='review-title']", "h3[class*='title']")
        pub_date  = self._first_text(card, "span[data-testid='review-date']", "p[data-testid='review-date']")
        pub_date  = re.sub(r'^Reviewed\s*:\s*', '', pub_date).strip()
        pub_date  = re.sub(r'^Rédigé le\s*:\s*', '', pub_date).strip()
        stay_date = self._first_text(card, "span[data-testid='review-stay-date']", "p.review_staydate")
        stay_date = re.sub(r'^Stayed\s*:\s*', '', stay_date).strip()
        stay_date = re.sub(r'^Séjour\s*:\s*', '', stay_date).strip()
        trip_type = self._first_text(
            card,
            "span[data-testid='review-traveler-type']",
            "p[data-testid='travel-type']",
            "div[data-testid='review-travel-type']",
        )
        pos  = self._text(card, "div[data-testid='review-positive-text']")
        neg  = self._text(card, "div[data-testid='review-negative-text']")
        full = (pos + " " + neg).strip()

        sub_scores = {}
        try:
            subs = card.find_elements(By.CSS_SELECTOR, "div[data-testid='review-subscore']")
            for s in subs:
                try:
                    cat = s.find_element(By.CSS_SELECTOR, "span").text.strip()
                    val = s.find_element(By.CSS_SELECTOR, "div.bc946a29db").text.strip()
                    sub_scores[cat] = self._extract_number(val)
                except Exception:
                    pass
        except Exception:
            pass

        return {
            "source": "booking.com",
            "hotel_name": hotel_name,
            "hotel_city": hotel_city,
            "reviewer_name": name,
            "nationality": nationality,
            "rating": rating,
            "review_title": title,
            "review_text": full,
            "positive_comment": pos,
            "negative_comment": neg,
            "publication_date": pub_date,
            "stay_date": stay_date,
            "trip_type": trip_type,
            "language": self.detect_language(full),
            **sub_scores,
        }

    def scrape_reviews(self, url, hotel_name, hotel_city, max_pages=90):
        print(f"🔄 Scraping : {hotel_name} ({hotel_city})")
        self.driver.get(url)
        time.sleep(6)

        all_reviews = []
        seen = set()
        page = 1

        while page <= max_pages:
            print(f"   📄 Page {page}  ({len(all_reviews)} avis cumulés)")

            try:
                self.wait.until(
                    EC.presence_of_element_located(
                        (By.CSS_SELECTOR, "div[data-testid='review-card']")
                    )
                )
            except Exception:
                print("   ⚠️  Aucune carte trouvée, fin du scraping.")
                break

            cards = self.driver.find_elements(
                By.CSS_SELECTOR, "div[data-testid='review-card']"
            )
            if not cards:
                print("   ⚠️  Liste de cartes vide, fin du scraping.")
                break

            snap = self._get_snapshot()

            new_count = 0
            for card in cards:
                review = self._parse_card(card, hotel_name, hotel_city)
                key = (
                    review.get("reviewer_name", ""),
                    review.get("publication_date", ""),
                    review.get("rating", ""),
                    review.get("review_text", "")[:50],
                )
                if key not in seen:
                    seen.add(key)
                    all_reviews.append(review)
                    new_count += 1

            print(f"      → {new_count} nouveaux avis")

            if new_count == 0:
                print("   ⚠️  Aucun nouvel avis — boucle détectée, arrêt.")
                break

            if not self._next_page(snap):
                print("   ✅ Dernière page atteinte.")
                break

            page += 1

        self.driver.quit()
        print(f"\n✅ {len(all_reviews)} avis extraits au total")
        return pd.DataFrame(all_reviews)


# ==================================================
if __name__ == "__main__":
    scraper = BookingScraper()
    df = scraper.scrape_reviews(URL, HOTEL_NAME, HOTEL_CITY, max_pages=MAX_PAGES)

    if not df.empty:
        filename = f"avis_{HOTEL_NAME.replace(' ', '_')}.csv"
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"\n💾 Fichier sauvegardé : {filename}  ({len(df)} avis)")

        print("\n🔍 Aperçu :")
        cols = [c for c in ["reviewer_name", "nationality", "rating",
                             "trip_type", "publication_date", "stay_date",
                             "language"] if c in df.columns]
        print(df[cols].head(10).to_string())

        print("\n📊 Colonnes vides :")
        empty = df[cols].isnull().sum() + (df[cols] == "").sum()
        print(empty[empty > 0] if empty[empty > 0].any() else "  Aucune colonne vide 🎉")
    else:
        print("❌ Aucun avis extrait.")

🔄 Scraping : Ibis Marrakech Centre Gare (Marrakech)
   📄 Page 1  (0 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 2  (10 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 3  (20 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 4  (30 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 5  (40 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 6  (50 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 7  (60 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tent

In [11]:
import time
import re
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from lingua import Language, LanguageDetectorBuilder

# ==================================================
URL = "https://www.booking.com/hotel/ma/ibis-moussafir-marrakech-palmeraie.html?aid=356980&label=gog235jc-10CAsojAFCImliaXMtbW91c3NhZmlyLW1hcnJha2VjaC1wYWxtZXJhaWVIDVgDaIwBiAEBmAEzuAEHyAEN2AED6AEB-AEBiAIBqAIBuAKC7aPPBsACAdICJDYwZDZmYTk5LTIxZjUtNDQ1Ni1hNGE5LWFjYTFjMDc0ZjRjYdgCAeACAQ&sid=4ba5cffa630e66002a1045ea6b678064&dist=0&keep_landing=1&sb_price_type=total&type=total&#tab-reviews"
HOTEL_NAME = "Ibis Marrakech Palmeraie"
HOTEL_CITY = "Marrakech"
MAX_PAGES = 200
# ==================================================


class BookingScraper:
    def __init__(self):
        options = Options()
        options.add_argument('--lang=en')
        options.add_argument('--disable-blink-features=AutomationControlled')
        options.add_experimental_option("excludeSwitches", ["enable-automation"])
        options.add_experimental_option('useAutomationExtension', False)
        options.add_argument(
            '--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
            'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
        )
        self.driver = webdriver.Chrome(
            service=Service(ChromeDriverManager().install()), options=options
        )
        self.wait = WebDriverWait(self.driver, 20)

        self.detector = LanguageDetectorBuilder.from_languages(
            Language.FRENCH, Language.ENGLISH, Language.GERMAN,
            Language.SPANISH, Language.ITALIAN, Language.PORTUGUESE,
            Language.DUTCH, Language.RUSSIAN, Language.ARABIC, Language.CHINESE,
        ).with_minimum_relative_distance(0.15).build()

        self._lang_map = {
            Language.FRENCH: "french", Language.ENGLISH: "english",
            Language.GERMAN: "german", Language.SPANISH: "spanish",
            Language.ITALIAN: "italian", Language.PORTUGUESE: "portuguese",
            Language.DUTCH: "dutch", Language.RUSSIAN: "russian",
            Language.ARABIC: "arabic", Language.CHINESE: "chinese",
        }

    def _text(self, parent, css):
        try:
            return parent.find_element(By.CSS_SELECTOR, css).text.strip()
        except Exception:
            return ""

    def _first_text(self, parent, *selectors):
        for sel in selectors:
            val = self._text(parent, sel)
            if val:
                return val
        return ""

    def _extract_number(self, raw):
        if not raw:
            return ""
        raw = raw.replace(",", ".")
        match = re.search(r'\d+(?:\.\d+)?', raw)
        return match.group(0) if match else ""

    def detect_language(self, text):
        if not text or len(text.strip()) < 5:
            return "unknown"
        try:
            lang = self.detector.detect_language_of(text)
            return self._lang_map.get(lang, "unknown") if lang else "unknown"
        except Exception:
            return "unknown"

    def _get_snapshot(self):
        try:
            card = self.driver.find_element(By.CSS_SELECTOR, "div[data-testid='review-card']")
            for sel in [
                "div[data-testid='review-positive-text']",
                "div[data-testid='review-negative-text']",
                "h4[data-testid='review-title']",
            ]:
                try:
                    t = card.find_element(By.CSS_SELECTOR, sel).text.strip()
                    if t:
                        return t
                except Exception:
                    pass
        except Exception:
            pass
        return ""

    def _next_page(self, snapshot_before):
        MAX_RETRIES = 3

        for attempt in range(1, MAX_RETRIES + 1):
            btn = None

            labels = ["Page suivante", "Suivant", "Next page", "Next"]
            for label in labels:
                try:
                    btn = self.driver.find_element(
                        By.CSS_SELECTOR, f"button[aria-label='{label}']"
                    )
                    print(f"      🔘 Bouton trouvé : '{label}'")
                    break
                except Exception:
                    pass

            if btn is None:
                try:
                    btn = self.driver.find_element(
                        By.XPATH,
                        "//button[contains(@aria-label,'suivante') or "
                        "contains(@aria-label,'Suivant') or "
                        "contains(@aria-label,'Next')]"
                    )
                    print(f"      🔘 Bouton via XPath : '{btn.get_attribute('aria-label')}'")
                except Exception:
                    pass

            if btn is None:
                print("   ⚠️  Bouton introuvable.")
                return False

            disabled      = btn.get_attribute("disabled")
            aria_disabled = btn.get_attribute("aria-disabled") or ""
            classes       = btn.get_attribute("class") or ""
            if disabled or aria_disabled == "true" or "disabled" in classes:
                print("   ✅ Bouton désactivé — dernière page atteinte.")
                return False

            try:
                self.driver.execute_script(
                    "arguments[0].scrollIntoView({block:'center'});", btn
                )
                time.sleep(1)
            except Exception:
                pass

            clicked = False
            for method in ['normal', 'js']:
                try:
                    if method == 'normal':
                        btn.click()
                    else:
                        self.driver.execute_script("arguments[0].click();", btn)
                    clicked = True
                    print(f"      ✅ Clic OK ({method}) — tentative {attempt}/{MAX_RETRIES}")
                    break
                except Exception as e:
                    print(f"      ⚠️  Clic {method} échoué : {e}")

            if not clicked:
                time.sleep(2)
                continue

            for _ in range(60):
                time.sleep(0.5)
                try:
                    cards = self.driver.find_elements(
                        By.CSS_SELECTOR, "div[data-testid='review-card']"
                    )
                    if not cards:
                        continue
                    first_text = ""
                    for sel in [
                        "div[data-testid='review-positive-text']",
                        "div[data-testid='review-negative-text']",
                        "h4[data-testid='review-title']",
                    ]:
                        try:
                            t = cards[0].find_element(By.CSS_SELECTOR, sel).text.strip()
                            if t:
                                first_text = t
                                break
                        except Exception:
                            pass
                    if first_text and first_text != snapshot_before:
                        return True
                except Exception:
                    pass

            print(f"   ⚠️  Contenu inchangé après 30s (tentative {attempt}/{MAX_RETRIES})")

            if attempt < MAX_RETRIES:
                print(f"   🔄 Nouvelle tentative dans 3s...")
                self.driver.execute_script("window.scrollTo(0, 0);")
                time.sleep(3)

        print("   ❌ Échec après 3 tentatives.")
        return False

    def _parse_card(self, card, hotel_name, hotel_city):
        name = self._first_text(
            card,
            "div.b08850ce41.f546354b44",
            "div[data-testid='reviewer-name'] span",
            "p.reviewer_name",
        )
        nationality = self._first_text(
            card,
            "span.d838fb5f41.aea5eccb71",
            "span[class*='reviewer_country']",
            "div[data-testid='reviewer-country'] span",
        )
        raw_rating = self._first_text(
            card,
            "div[data-testid='review-score'] div.bc946a29db",
            "div[data-testid='review-score']",
            "div.bui-review-score__badge",
        )
        rating    = self._extract_number(raw_rating)
        title     = self._first_text(card, "h4[data-testid='review-title']", "h3[class*='title']")
        pub_date  = self._first_text(card, "span[data-testid='review-date']", "p[data-testid='review-date']")
        pub_date  = re.sub(r'^Reviewed\s*:\s*', '', pub_date).strip()
        pub_date  = re.sub(r'^Rédigé le\s*:\s*', '', pub_date).strip()
        stay_date = self._first_text(card, "span[data-testid='review-stay-date']", "p.review_staydate")
        stay_date = re.sub(r'^Stayed\s*:\s*', '', stay_date).strip()
        stay_date = re.sub(r'^Séjour\s*:\s*', '', stay_date).strip()
        trip_type = self._first_text(
            card,
            "span[data-testid='review-traveler-type']",
            "p[data-testid='travel-type']",
            "div[data-testid='review-travel-type']",
        )
        pos  = self._text(card, "div[data-testid='review-positive-text']")
        neg  = self._text(card, "div[data-testid='review-negative-text']")
        full = (pos + " " + neg).strip()

        sub_scores = {}
        try:
            subs = card.find_elements(By.CSS_SELECTOR, "div[data-testid='review-subscore']")
            for s in subs:
                try:
                    cat = s.find_element(By.CSS_SELECTOR, "span").text.strip()
                    val = s.find_element(By.CSS_SELECTOR, "div.bc946a29db").text.strip()
                    sub_scores[cat] = self._extract_number(val)
                except Exception:
                    pass
        except Exception:
            pass

        return {
            "source": "booking.com",
            "hotel_name": hotel_name,
            "hotel_city": hotel_city,
            "reviewer_name": name,
            "nationality": nationality,
            "rating": rating,
            "review_title": title,
            "review_text": full,
            "positive_comment": pos,
            "negative_comment": neg,
            "publication_date": pub_date,
            "stay_date": stay_date,
            "trip_type": trip_type,
            "language": self.detect_language(full),
            **sub_scores,
        }

    def scrape_reviews(self, url, hotel_name, hotel_city, max_pages=90):
        print(f"🔄 Scraping : {hotel_name} ({hotel_city})")
        self.driver.get(url)
        time.sleep(6)

        all_reviews = []
        seen = set()
        page = 1

        while page <= max_pages:
            print(f"   📄 Page {page}  ({len(all_reviews)} avis cumulés)")

            try:
                self.wait.until(
                    EC.presence_of_element_located(
                        (By.CSS_SELECTOR, "div[data-testid='review-card']")
                    )
                )
            except Exception:
                print("   ⚠️  Aucune carte trouvée, fin du scraping.")
                break

            cards = self.driver.find_elements(
                By.CSS_SELECTOR, "div[data-testid='review-card']"
            )
            if not cards:
                print("   ⚠️  Liste de cartes vide, fin du scraping.")
                break

            snap = self._get_snapshot()

            new_count = 0
            for card in cards:
                review = self._parse_card(card, hotel_name, hotel_city)
                key = (
                    review.get("reviewer_name", ""),
                    review.get("publication_date", ""),
                    review.get("rating", ""),
                    review.get("review_text", "")[:50],
                )
                if key not in seen:
                    seen.add(key)
                    all_reviews.append(review)
                    new_count += 1

            print(f"      → {new_count} nouveaux avis")

            if new_count == 0:
                print("   ⚠️  Aucun nouvel avis — boucle détectée, arrêt.")
                break

            if not self._next_page(snap):
                print("   ✅ Dernière page atteinte.")
                break

            page += 1

        self.driver.quit()
        print(f"\n✅ {len(all_reviews)} avis extraits au total")
        return pd.DataFrame(all_reviews)


# ==================================================
if __name__ == "__main__":
    scraper = BookingScraper()
    df = scraper.scrape_reviews(URL, HOTEL_NAME, HOTEL_CITY, max_pages=MAX_PAGES)

    if not df.empty:
        filename = f"avis_{HOTEL_NAME.replace(' ', '_')}.csv"
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"\n💾 Fichier sauvegardé : {filename}  ({len(df)} avis)")

        print("\n🔍 Aperçu :")
        cols = [c for c in ["reviewer_name", "nationality", "rating",
                             "trip_type", "publication_date", "stay_date",
                             "language"] if c in df.columns]
        print(df[cols].head(10).to_string())

        print("\n📊 Colonnes vides :")
        empty = df[cols].isnull().sum() + (df[cols] == "").sum()
        print(empty[empty > 0] if empty[empty > 0].any() else "  Aucune colonne vide 🎉")
    else:
        print("❌ Aucun avis extrait.")

🔄 Scraping : Ibis Marrakech Palmeraie (Marrakech)
   📄 Page 1  (0 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 2  (10 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 3  (20 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 4  (30 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 5  (40 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 6  (50 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 7  (60 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentat

In [13]:
import time
import re
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from lingua import Language, LanguageDetectorBuilder

# ==================================================
URL = "https://www.booking.com/hotel/ma/ibis-moussafir-meknes.html?aid=356980&label=gog235jc-10CAsojAFCFWliaXMtbW91c3NhZmlyLW1la25lc0gNWANojAGIAQGYATO4AQfIAQzYAQPoAQH4AQGIAgGoAgG4AuWApM8GwAIB0gIkZDBjODQ3NmEtNDAwYS00OTZkLTgyZDItNDlmMDU5YTU2ZjA42AIB4AIB&sid=53812c21f587e9070627ab5a462ac31d&dest_id=-39119&dest_type=city&dist=0&group_adults=2&group_children=0&hapos=1&hpos=1&no_rooms=1&req_adults=2&req_children=0&room1=A%2CA&sb_price_type=total&sr_order=popularity&srepoch=1776877683&srpvid=4f43787355a2074e&type=total&ucfs=1&lang=en-us&soz=1&lang_changed=1#tab-reviews"
HOTEL_NAME = "Ibis Meknes"
HOTEL_CITY = "Meknes"
MAX_PAGES = 120
# ==================================================


class BookingScraper:
    def __init__(self):
        options = Options()
        options.add_argument('--lang=en')
        options.add_argument('--disable-blink-features=AutomationControlled')
        options.add_experimental_option("excludeSwitches", ["enable-automation"])
        options.add_experimental_option('useAutomationExtension', False)
        options.add_argument(
            '--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
            'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
        )
        self.driver = webdriver.Chrome(
            service=Service(ChromeDriverManager().install()), options=options
        )
        self.wait = WebDriverWait(self.driver, 20)

        self.detector = LanguageDetectorBuilder.from_languages(
            Language.FRENCH, Language.ENGLISH, Language.GERMAN,
            Language.SPANISH, Language.ITALIAN, Language.PORTUGUESE,
            Language.DUTCH, Language.RUSSIAN, Language.ARABIC, Language.CHINESE,
        ).with_minimum_relative_distance(0.15).build()

        self._lang_map = {
            Language.FRENCH: "french", Language.ENGLISH: "english",
            Language.GERMAN: "german", Language.SPANISH: "spanish",
            Language.ITALIAN: "italian", Language.PORTUGUESE: "portuguese",
            Language.DUTCH: "dutch", Language.RUSSIAN: "russian",
            Language.ARABIC: "arabic", Language.CHINESE: "chinese",
        }

    def _text(self, parent, css):
        try:
            return parent.find_element(By.CSS_SELECTOR, css).text.strip()
        except Exception:
            return ""

    def _first_text(self, parent, *selectors):
        for sel in selectors:
            val = self._text(parent, sel)
            if val:
                return val
        return ""

    def _extract_number(self, raw):
        if not raw:
            return ""
        raw = raw.replace(",", ".")
        match = re.search(r'\d+(?:\.\d+)?', raw)
        return match.group(0) if match else ""

    def detect_language(self, text):
        if not text or len(text.strip()) < 5:
            return "unknown"
        try:
            lang = self.detector.detect_language_of(text)
            return self._lang_map.get(lang, "unknown") if lang else "unknown"
        except Exception:
            return "unknown"

    def _get_snapshot(self):
        try:
            card = self.driver.find_element(By.CSS_SELECTOR, "div[data-testid='review-card']")
            for sel in [
                "div[data-testid='review-positive-text']",
                "div[data-testid='review-negative-text']",
                "h4[data-testid='review-title']",
            ]:
                try:
                    t = card.find_element(By.CSS_SELECTOR, sel).text.strip()
                    if t:
                        return t
                except Exception:
                    pass
        except Exception:
            pass
        return ""

    def _next_page(self, snapshot_before):
        MAX_RETRIES = 3

        for attempt in range(1, MAX_RETRIES + 1):
            btn = None

            labels = ["Page suivante", "Suivant", "Next page", "Next"]
            for label in labels:
                try:
                    btn = self.driver.find_element(
                        By.CSS_SELECTOR, f"button[aria-label='{label}']"
                    )
                    print(f"      🔘 Bouton trouvé : '{label}'")
                    break
                except Exception:
                    pass

            if btn is None:
                try:
                    btn = self.driver.find_element(
                        By.XPATH,
                        "//button[contains(@aria-label,'suivante') or "
                        "contains(@aria-label,'Suivant') or "
                        "contains(@aria-label,'Next')]"
                    )
                    print(f"      🔘 Bouton via XPath : '{btn.get_attribute('aria-label')}'")
                except Exception:
                    pass

            if btn is None:
                print("   ⚠️  Bouton introuvable.")
                return False

            disabled      = btn.get_attribute("disabled")
            aria_disabled = btn.get_attribute("aria-disabled") or ""
            classes       = btn.get_attribute("class") or ""
            if disabled or aria_disabled == "true" or "disabled" in classes:
                print("   ✅ Bouton désactivé — dernière page atteinte.")
                return False

            try:
                self.driver.execute_script(
                    "arguments[0].scrollIntoView({block:'center'});", btn
                )
                time.sleep(1)
            except Exception:
                pass

            clicked = False
            for method in ['normal', 'js']:
                try:
                    if method == 'normal':
                        btn.click()
                    else:
                        self.driver.execute_script("arguments[0].click();", btn)
                    clicked = True
                    print(f"      ✅ Clic OK ({method}) — tentative {attempt}/{MAX_RETRIES}")
                    break
                except Exception as e:
                    print(f"      ⚠️  Clic {method} échoué : {e}")

            if not clicked:
                time.sleep(2)
                continue

            for _ in range(60):
                time.sleep(0.5)
                try:
                    cards = self.driver.find_elements(
                        By.CSS_SELECTOR, "div[data-testid='review-card']"
                    )
                    if not cards:
                        continue
                    first_text = ""
                    for sel in [
                        "div[data-testid='review-positive-text']",
                        "div[data-testid='review-negative-text']",
                        "h4[data-testid='review-title']",
                    ]:
                        try:
                            t = cards[0].find_element(By.CSS_SELECTOR, sel).text.strip()
                            if t:
                                first_text = t
                                break
                        except Exception:
                            pass
                    if first_text and first_text != snapshot_before:
                        return True
                except Exception:
                    pass

            print(f"   ⚠️  Contenu inchangé après 30s (tentative {attempt}/{MAX_RETRIES})")

            if attempt < MAX_RETRIES:
                print(f"   🔄 Nouvelle tentative dans 3s...")
                self.driver.execute_script("window.scrollTo(0, 0);")
                time.sleep(3)

        print("   ❌ Échec après 3 tentatives.")
        return False

    def _parse_card(self, card, hotel_name, hotel_city):
        name = self._first_text(
            card,
            "div.b08850ce41.f546354b44",
            "div[data-testid='reviewer-name'] span",
            "p.reviewer_name",
        )
        nationality = self._first_text(
            card,
            "span.d838fb5f41.aea5eccb71",
            "span[class*='reviewer_country']",
            "div[data-testid='reviewer-country'] span",
        )
        raw_rating = self._first_text(
            card,
            "div[data-testid='review-score'] div.bc946a29db",
            "div[data-testid='review-score']",
            "div.bui-review-score__badge",
        )
        rating    = self._extract_number(raw_rating)
        title     = self._first_text(card, "h4[data-testid='review-title']", "h3[class*='title']")
        pub_date  = self._first_text(card, "span[data-testid='review-date']", "p[data-testid='review-date']")
        pub_date  = re.sub(r'^Reviewed\s*:\s*', '', pub_date).strip()
        pub_date  = re.sub(r'^Rédigé le\s*:\s*', '', pub_date).strip()
        stay_date = self._first_text(card, "span[data-testid='review-stay-date']", "p.review_staydate")
        stay_date = re.sub(r'^Stayed\s*:\s*', '', stay_date).strip()
        stay_date = re.sub(r'^Séjour\s*:\s*', '', stay_date).strip()
        trip_type = self._first_text(
            card,
            "span[data-testid='review-traveler-type']",
            "p[data-testid='travel-type']",
            "div[data-testid='review-travel-type']",
        )
        pos  = self._text(card, "div[data-testid='review-positive-text']")
        neg  = self._text(card, "div[data-testid='review-negative-text']")
        full = (pos + " " + neg).strip()

        sub_scores = {}
        try:
            subs = card.find_elements(By.CSS_SELECTOR, "div[data-testid='review-subscore']")
            for s in subs:
                try:
                    cat = s.find_element(By.CSS_SELECTOR, "span").text.strip()
                    val = s.find_element(By.CSS_SELECTOR, "div.bc946a29db").text.strip()
                    sub_scores[cat] = self._extract_number(val)
                except Exception:
                    pass
        except Exception:
            pass

        return {
            "source": "booking.com",
            "hotel_name": hotel_name,
            "hotel_city": hotel_city,
            "reviewer_name": name,
            "nationality": nationality,
            "rating": rating,
            "review_title": title,
            "review_text": full,
            "positive_comment": pos,
            "negative_comment": neg,
            "publication_date": pub_date,
            "stay_date": stay_date,
            "trip_type": trip_type,
            "language": self.detect_language(full),
            **sub_scores,
        }

    def scrape_reviews(self, url, hotel_name, hotel_city, max_pages=90):
        print(f"🔄 Scraping : {hotel_name} ({hotel_city})")
        self.driver.get(url)
        time.sleep(6)

        all_reviews = []
        seen = set()
        page = 1

        while page <= max_pages:
            print(f"   📄 Page {page}  ({len(all_reviews)} avis cumulés)")

            try:
                self.wait.until(
                    EC.presence_of_element_located(
                        (By.CSS_SELECTOR, "div[data-testid='review-card']")
                    )
                )
            except Exception:
                print("   ⚠️  Aucune carte trouvée, fin du scraping.")
                break

            cards = self.driver.find_elements(
                By.CSS_SELECTOR, "div[data-testid='review-card']"
            )
            if not cards:
                print("   ⚠️  Liste de cartes vide, fin du scraping.")
                break

            snap = self._get_snapshot()

            new_count = 0
            for card in cards:
                review = self._parse_card(card, hotel_name, hotel_city)
                key = (
                    review.get("reviewer_name", ""),
                    review.get("publication_date", ""),
                    review.get("rating", ""),
                    review.get("review_text", "")[:50],
                )
                if key not in seen:
                    seen.add(key)
                    all_reviews.append(review)
                    new_count += 1

            print(f"      → {new_count} nouveaux avis")

            if new_count == 0:
                print("   ⚠️  Aucun nouvel avis — boucle détectée, arrêt.")
                break

            if not self._next_page(snap):
                print("   ✅ Dernière page atteinte.")
                break

            page += 1

        self.driver.quit()
        print(f"\n✅ {len(all_reviews)} avis extraits au total")
        return pd.DataFrame(all_reviews)


# ==================================================
if __name__ == "__main__":
    scraper = BookingScraper()
    df = scraper.scrape_reviews(URL, HOTEL_NAME, HOTEL_CITY, max_pages=MAX_PAGES)

    if not df.empty:
        filename = f"avis_{HOTEL_NAME.replace(' ', '_')}.csv"
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"\n💾 Fichier sauvegardé : {filename}  ({len(df)} avis)")

        print("\n🔍 Aperçu :")
        cols = [c for c in ["reviewer_name", "nationality", "rating",
                             "trip_type", "publication_date", "stay_date",
                             "language"] if c in df.columns]
        print(df[cols].head(10).to_string())

        print("\n📊 Colonnes vides :")
        empty = df[cols].isnull().sum() + (df[cols] == "").sum()
        print(empty[empty > 0] if empty[empty > 0].any() else "  Aucune colonne vide 🎉")
    else:
        print("❌ Aucun avis extrait.")

🔄 Scraping : Ibis Meknes (Meknes)
   📄 Page 1  (0 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Next page'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 2  (10 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Next page'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 3  (20 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Next page'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 4  (30 avis cumulés)
      → 9 nouveaux avis
      🔘 Bouton trouvé : 'Next page'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 5  (39 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Next page'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 6  (49 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Next page'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 7  (59 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Next page'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 8  (69 avis cumulés)
      

In [15]:
import time
import re
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from lingua import Language, LanguageDetectorBuilder

# ==================================================
URL = "https://www.booking.com/hotel/ma/ibis-moussafir-oujda.html?aid=356980&label=gog235jc-10CAsojAFCFWliaXMtbW91c3NhZmlyLW1la25lc0gNWANojAGIAQGYATO4AQfIAQzYAQPoAQH4AQGIAgGoAgG4AuWApM8GwAIB0gIkZDBjODQ3NmEtNDAwYS00OTZkLTgyZDItNDlmMDU5YTU2ZjA42AIB4AIB&sid=53812c21f587e9070627ab5a462ac31d&dest_id=98912&dest_type=hotel&dist=0&group_adults=2&group_children=0&hapos=1&hpos=1&no_rooms=1&req_adults=2&req_children=0&room1=A%2CA&sb_price_type=total&sr_order=popularity&srepoch=1776879998&srpvid=8c427cf522e50d6e&type=total&ucfs=1&#tab-reviews"
HOTEL_NAME = "Ibis Oujda"
HOTEL_CITY = "Oujda"
MAX_PAGES = 80
# ==================================================


class BookingScraper:
    def __init__(self):
        options = Options()
        options.add_argument('--lang=en')
        options.add_argument('--disable-blink-features=AutomationControlled')
        options.add_experimental_option("excludeSwitches", ["enable-automation"])
        options.add_experimental_option('useAutomationExtension', False)
        options.add_argument(
            '--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
            'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
        )
        self.driver = webdriver.Chrome(
            service=Service(ChromeDriverManager().install()), options=options
        )
        self.wait = WebDriverWait(self.driver, 20)

        self.detector = LanguageDetectorBuilder.from_languages(
            Language.FRENCH, Language.ENGLISH, Language.GERMAN,
            Language.SPANISH, Language.ITALIAN, Language.PORTUGUESE,
            Language.DUTCH, Language.RUSSIAN, Language.ARABIC, Language.CHINESE,
        ).with_minimum_relative_distance(0.15).build()

        self._lang_map = {
            Language.FRENCH: "french", Language.ENGLISH: "english",
            Language.GERMAN: "german", Language.SPANISH: "spanish",
            Language.ITALIAN: "italian", Language.PORTUGUESE: "portuguese",
            Language.DUTCH: "dutch", Language.RUSSIAN: "russian",
            Language.ARABIC: "arabic", Language.CHINESE: "chinese",
        }

    def _text(self, parent, css):
        try:
            return parent.find_element(By.CSS_SELECTOR, css).text.strip()
        except Exception:
            return ""

    def _first_text(self, parent, *selectors):
        for sel in selectors:
            val = self._text(parent, sel)
            if val:
                return val
        return ""

    def _extract_number(self, raw):
        if not raw:
            return ""
        raw = raw.replace(",", ".")
        match = re.search(r'\d+(?:\.\d+)?', raw)
        return match.group(0) if match else ""

    def detect_language(self, text):
        if not text or len(text.strip()) < 5:
            return "unknown"
        try:
            lang = self.detector.detect_language_of(text)
            return self._lang_map.get(lang, "unknown") if lang else "unknown"
        except Exception:
            return "unknown"

    def _get_snapshot(self):
        try:
            card = self.driver.find_element(By.CSS_SELECTOR, "div[data-testid='review-card']")
            for sel in [
                "div[data-testid='review-positive-text']",
                "div[data-testid='review-negative-text']",
                "h4[data-testid='review-title']",
            ]:
                try:
                    t = card.find_element(By.CSS_SELECTOR, sel).text.strip()
                    if t:
                        return t
                except Exception:
                    pass
        except Exception:
            pass
        return ""

    def _next_page(self, snapshot_before):
        MAX_RETRIES = 3

        for attempt in range(1, MAX_RETRIES + 1):
            btn = None

            labels = ["Page suivante", "Suivant", "Next page", "Next"]
            for label in labels:
                try:
                    btn = self.driver.find_element(
                        By.CSS_SELECTOR, f"button[aria-label='{label}']"
                    )
                    print(f"      🔘 Bouton trouvé : '{label}'")
                    break
                except Exception:
                    pass

            if btn is None:
                try:
                    btn = self.driver.find_element(
                        By.XPATH,
                        "//button[contains(@aria-label,'suivante') or "
                        "contains(@aria-label,'Suivant') or "
                        "contains(@aria-label,'Next')]"
                    )
                    print(f"      🔘 Bouton via XPath : '{btn.get_attribute('aria-label')}'")
                except Exception:
                    pass

            if btn is None:
                print("   ⚠️  Bouton introuvable.")
                return False

            disabled      = btn.get_attribute("disabled")
            aria_disabled = btn.get_attribute("aria-disabled") or ""
            classes       = btn.get_attribute("class") or ""
            if disabled or aria_disabled == "true" or "disabled" in classes:
                print("   ✅ Bouton désactivé — dernière page atteinte.")
                return False

            try:
                self.driver.execute_script(
                    "arguments[0].scrollIntoView({block:'center'});", btn
                )
                time.sleep(1)
            except Exception:
                pass

            clicked = False
            for method in ['normal', 'js']:
                try:
                    if method == 'normal':
                        btn.click()
                    else:
                        self.driver.execute_script("arguments[0].click();", btn)
                    clicked = True
                    print(f"      ✅ Clic OK ({method}) — tentative {attempt}/{MAX_RETRIES}")
                    break
                except Exception as e:
                    print(f"      ⚠️  Clic {method} échoué : {e}")

            if not clicked:
                time.sleep(2)
                continue

            for _ in range(60):
                time.sleep(0.5)
                try:
                    cards = self.driver.find_elements(
                        By.CSS_SELECTOR, "div[data-testid='review-card']"
                    )
                    if not cards:
                        continue
                    first_text = ""
                    for sel in [
                        "div[data-testid='review-positive-text']",
                        "div[data-testid='review-negative-text']",
                        "h4[data-testid='review-title']",
                    ]:
                        try:
                            t = cards[0].find_element(By.CSS_SELECTOR, sel).text.strip()
                            if t:
                                first_text = t
                                break
                        except Exception:
                            pass
                    if first_text and first_text != snapshot_before:
                        return True
                except Exception:
                    pass

            print(f"   ⚠️  Contenu inchangé après 30s (tentative {attempt}/{MAX_RETRIES})")

            if attempt < MAX_RETRIES:
                print(f"   🔄 Nouvelle tentative dans 3s...")
                self.driver.execute_script("window.scrollTo(0, 0);")
                time.sleep(3)

        print("   ❌ Échec après 3 tentatives.")
        return False

    def _parse_card(self, card, hotel_name, hotel_city):
        name = self._first_text(
            card,
            "div.b08850ce41.f546354b44",
            "div[data-testid='reviewer-name'] span",
            "p.reviewer_name",
        )
        nationality = self._first_text(
            card,
            "span.d838fb5f41.aea5eccb71",
            "span[class*='reviewer_country']",
            "div[data-testid='reviewer-country'] span",
        )
        raw_rating = self._first_text(
            card,
            "div[data-testid='review-score'] div.bc946a29db",
            "div[data-testid='review-score']",
            "div.bui-review-score__badge",
        )
        rating    = self._extract_number(raw_rating)
        title     = self._first_text(card, "h4[data-testid='review-title']", "h3[class*='title']")
        pub_date  = self._first_text(card, "span[data-testid='review-date']", "p[data-testid='review-date']")
        pub_date  = re.sub(r'^Reviewed\s*:\s*', '', pub_date).strip()
        pub_date  = re.sub(r'^Rédigé le\s*:\s*', '', pub_date).strip()
        stay_date = self._first_text(card, "span[data-testid='review-stay-date']", "p.review_staydate")
        stay_date = re.sub(r'^Stayed\s*:\s*', '', stay_date).strip()
        stay_date = re.sub(r'^Séjour\s*:\s*', '', stay_date).strip()
        trip_type = self._first_text(
            card,
            "span[data-testid='review-traveler-type']",
            "p[data-testid='travel-type']",
            "div[data-testid='review-travel-type']",
        )
        pos  = self._text(card, "div[data-testid='review-positive-text']")
        neg  = self._text(card, "div[data-testid='review-negative-text']")
        full = (pos + " " + neg).strip()

        sub_scores = {}
        try:
            subs = card.find_elements(By.CSS_SELECTOR, "div[data-testid='review-subscore']")
            for s in subs:
                try:
                    cat = s.find_element(By.CSS_SELECTOR, "span").text.strip()
                    val = s.find_element(By.CSS_SELECTOR, "div.bc946a29db").text.strip()
                    sub_scores[cat] = self._extract_number(val)
                except Exception:
                    pass
        except Exception:
            pass

        return {
            "source": "booking.com",
            "hotel_name": hotel_name,
            "hotel_city": hotel_city,
            "reviewer_name": name,
            "nationality": nationality,
            "rating": rating,
            "review_title": title,
            "review_text": full,
            "positive_comment": pos,
            "negative_comment": neg,
            "publication_date": pub_date,
            "stay_date": stay_date,
            "trip_type": trip_type,
            "language": self.detect_language(full),
            **sub_scores,
        }

    def scrape_reviews(self, url, hotel_name, hotel_city, max_pages=90):
        print(f"🔄 Scraping : {hotel_name} ({hotel_city})")
        self.driver.get(url)
        time.sleep(6)

        all_reviews = []
        seen = set()
        page = 1

        while page <= max_pages:
            print(f"   📄 Page {page}  ({len(all_reviews)} avis cumulés)")

            try:
                self.wait.until(
                    EC.presence_of_element_located(
                        (By.CSS_SELECTOR, "div[data-testid='review-card']")
                    )
                )
            except Exception:
                print("   ⚠️  Aucune carte trouvée, fin du scraping.")
                break

            cards = self.driver.find_elements(
                By.CSS_SELECTOR, "div[data-testid='review-card']"
            )
            if not cards:
                print("   ⚠️  Liste de cartes vide, fin du scraping.")
                break

            snap = self._get_snapshot()

            new_count = 0
            for card in cards:
                review = self._parse_card(card, hotel_name, hotel_city)
                key = (
                    review.get("reviewer_name", ""),
                    review.get("publication_date", ""),
                    review.get("rating", ""),
                    review.get("review_text", "")[:50],
                )
                if key not in seen:
                    seen.add(key)
                    all_reviews.append(review)
                    new_count += 1

            print(f"      → {new_count} nouveaux avis")

            if new_count == 0:
                print("   ⚠️  Aucun nouvel avis — boucle détectée, arrêt.")
                break

            if not self._next_page(snap):
                print("   ✅ Dernière page atteinte.")
                break

            page += 1

        self.driver.quit()
        print(f"\n✅ {len(all_reviews)} avis extraits au total")
        return pd.DataFrame(all_reviews)


# ==================================================
if __name__ == "__main__":
    scraper = BookingScraper()
    df = scraper.scrape_reviews(URL, HOTEL_NAME, HOTEL_CITY, max_pages=MAX_PAGES)

    if not df.empty:
        filename = f"avis_{HOTEL_NAME.replace(' ', '_')}.csv"
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"\n💾 Fichier sauvegardé : {filename}  ({len(df)} avis)")

        print("\n🔍 Aperçu :")
        cols = [c for c in ["reviewer_name", "nationality", "rating",
                             "trip_type", "publication_date", "stay_date",
                             "language"] if c in df.columns]
        print(df[cols].head(10).to_string())

        print("\n📊 Colonnes vides :")
        empty = df[cols].isnull().sum() + (df[cols] == "").sum()
        print(empty[empty > 0] if empty[empty > 0].any() else "  Aucune colonne vide 🎉")
    else:
        print("❌ Aucun avis extrait.")

🔄 Scraping : Ibis Oujda (Oujda)
   📄 Page 1  (0 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 2  (10 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 3  (20 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 4  (30 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 5  (40 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 6  (50 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 7  (60 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 

In [16]:
import time
import re
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from lingua import Language, LanguageDetectorBuilder

# ==================================================
URL = "https://www.booking.com/hotel/ma/ibis-mohammedia.html?aid=356980&label=gog235jc-10CAsojAFCD2liaXMtbW9oYW1tZWRpYUgNWANojAGIAQGYATO4AQfIAQ3YAQPoAQH4AQGIAgGoAgG4Ao2mpM8GwAIB0gIkOTNjZjE4YmEtMGYzYS00NzQ1LWE1ODgtNDJkZDE4NWVmODM32AIB4AIB&sid=4ba5cffa630e66002a1045ea6b678064&dist=0&keep_landing=1&sb_price_type=total&type=total&#tab-reviews"
HOTEL_NAME = "Ibis Mohammedia"
HOTEL_CITY = "Mohammedia"
MAX_PAGES = 120
# ==================================================


class BookingScraper:
    def __init__(self):
        options = Options()
        options.add_argument('--lang=en')
        options.add_argument('--disable-blink-features=AutomationControlled')
        options.add_experimental_option("excludeSwitches", ["enable-automation"])
        options.add_experimental_option('useAutomationExtension', False)
        options.add_argument(
            '--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
            'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
        )
        self.driver = webdriver.Chrome(
            service=Service(ChromeDriverManager().install()), options=options
        )
        self.wait = WebDriverWait(self.driver, 20)

        self.detector = LanguageDetectorBuilder.from_languages(
            Language.FRENCH, Language.ENGLISH, Language.GERMAN,
            Language.SPANISH, Language.ITALIAN, Language.PORTUGUESE,
            Language.DUTCH, Language.RUSSIAN, Language.ARABIC, Language.CHINESE,
        ).with_minimum_relative_distance(0.15).build()

        self._lang_map = {
            Language.FRENCH: "french", Language.ENGLISH: "english",
            Language.GERMAN: "german", Language.SPANISH: "spanish",
            Language.ITALIAN: "italian", Language.PORTUGUESE: "portuguese",
            Language.DUTCH: "dutch", Language.RUSSIAN: "russian",
            Language.ARABIC: "arabic", Language.CHINESE: "chinese",
        }

    def _text(self, parent, css):
        try:
            return parent.find_element(By.CSS_SELECTOR, css).text.strip()
        except Exception:
            return ""

    def _first_text(self, parent, *selectors):
        for sel in selectors:
            val = self._text(parent, sel)
            if val:
                return val
        return ""

    def _extract_number(self, raw):
        if not raw:
            return ""
        raw = raw.replace(",", ".")
        match = re.search(r'\d+(?:\.\d+)?', raw)
        return match.group(0) if match else ""

    def detect_language(self, text):
        if not text or len(text.strip()) < 5:
            return "unknown"
        try:
            lang = self.detector.detect_language_of(text)
            return self._lang_map.get(lang, "unknown") if lang else "unknown"
        except Exception:
            return "unknown"

    def _get_snapshot(self):
        try:
            card = self.driver.find_element(By.CSS_SELECTOR, "div[data-testid='review-card']")
            for sel in [
                "div[data-testid='review-positive-text']",
                "div[data-testid='review-negative-text']",
                "h4[data-testid='review-title']",
            ]:
                try:
                    t = card.find_element(By.CSS_SELECTOR, sel).text.strip()
                    if t:
                        return t
                except Exception:
                    pass
        except Exception:
            pass
        return ""

    def _next_page(self, snapshot_before):
        MAX_RETRIES = 3

        for attempt in range(1, MAX_RETRIES + 1):
            btn = None

            labels = ["Page suivante", "Suivant", "Next page", "Next"]
            for label in labels:
                try:
                    btn = self.driver.find_element(
                        By.CSS_SELECTOR, f"button[aria-label='{label}']"
                    )
                    print(f"      🔘 Bouton trouvé : '{label}'")
                    break
                except Exception:
                    pass

            if btn is None:
                try:
                    btn = self.driver.find_element(
                        By.XPATH,
                        "//button[contains(@aria-label,'suivante') or "
                        "contains(@aria-label,'Suivant') or "
                        "contains(@aria-label,'Next')]"
                    )
                    print(f"      🔘 Bouton via XPath : '{btn.get_attribute('aria-label')}'")
                except Exception:
                    pass

            if btn is None:
                print("   ⚠️  Bouton introuvable.")
                return False

            disabled      = btn.get_attribute("disabled")
            aria_disabled = btn.get_attribute("aria-disabled") or ""
            classes       = btn.get_attribute("class") or ""
            if disabled or aria_disabled == "true" or "disabled" in classes:
                print("   ✅ Bouton désactivé — dernière page atteinte.")
                return False

            try:
                self.driver.execute_script(
                    "arguments[0].scrollIntoView({block:'center'});", btn
                )
                time.sleep(1)
            except Exception:
                pass

            clicked = False
            for method in ['normal', 'js']:
                try:
                    if method == 'normal':
                        btn.click()
                    else:
                        self.driver.execute_script("arguments[0].click();", btn)
                    clicked = True
                    print(f"      ✅ Clic OK ({method}) — tentative {attempt}/{MAX_RETRIES}")
                    break
                except Exception as e:
                    print(f"      ⚠️  Clic {method} échoué : {e}")

            if not clicked:
                time.sleep(2)
                continue

            for _ in range(60):
                time.sleep(0.5)
                try:
                    cards = self.driver.find_elements(
                        By.CSS_SELECTOR, "div[data-testid='review-card']"
                    )
                    if not cards:
                        continue
                    first_text = ""
                    for sel in [
                        "div[data-testid='review-positive-text']",
                        "div[data-testid='review-negative-text']",
                        "h4[data-testid='review-title']",
                    ]:
                        try:
                            t = cards[0].find_element(By.CSS_SELECTOR, sel).text.strip()
                            if t:
                                first_text = t
                                break
                        except Exception:
                            pass
                    if first_text and first_text != snapshot_before:
                        return True
                except Exception:
                    pass

            print(f"   ⚠️  Contenu inchangé après 30s (tentative {attempt}/{MAX_RETRIES})")

            if attempt < MAX_RETRIES:
                print(f"   🔄 Nouvelle tentative dans 3s...")
                self.driver.execute_script("window.scrollTo(0, 0);")
                time.sleep(3)

        print("   ❌ Échec après 3 tentatives.")
        return False

    def _parse_card(self, card, hotel_name, hotel_city):
        name = self._first_text(
            card,
            "div.b08850ce41.f546354b44",
            "div[data-testid='reviewer-name'] span",
            "p.reviewer_name",
        )
        nationality = self._first_text(
            card,
            "span.d838fb5f41.aea5eccb71",
            "span[class*='reviewer_country']",
            "div[data-testid='reviewer-country'] span",
        )
        raw_rating = self._first_text(
            card,
            "div[data-testid='review-score'] div.bc946a29db",
            "div[data-testid='review-score']",
            "div.bui-review-score__badge",
        )
        rating    = self._extract_number(raw_rating)
        title     = self._first_text(card, "h4[data-testid='review-title']", "h3[class*='title']")
        pub_date  = self._first_text(card, "span[data-testid='review-date']", "p[data-testid='review-date']")
        pub_date  = re.sub(r'^Reviewed\s*:\s*', '', pub_date).strip()
        pub_date  = re.sub(r'^Rédigé le\s*:\s*', '', pub_date).strip()
        stay_date = self._first_text(card, "span[data-testid='review-stay-date']", "p.review_staydate")
        stay_date = re.sub(r'^Stayed\s*:\s*', '', stay_date).strip()
        stay_date = re.sub(r'^Séjour\s*:\s*', '', stay_date).strip()
        trip_type = self._first_text(
            card,
            "span[data-testid='review-traveler-type']",
            "p[data-testid='travel-type']",
            "div[data-testid='review-travel-type']",
        )
        pos  = self._text(card, "div[data-testid='review-positive-text']")
        neg  = self._text(card, "div[data-testid='review-negative-text']")
        full = (pos + " " + neg).strip()

        sub_scores = {}
        try:
            subs = card.find_elements(By.CSS_SELECTOR, "div[data-testid='review-subscore']")
            for s in subs:
                try:
                    cat = s.find_element(By.CSS_SELECTOR, "span").text.strip()
                    val = s.find_element(By.CSS_SELECTOR, "div.bc946a29db").text.strip()
                    sub_scores[cat] = self._extract_number(val)
                except Exception:
                    pass
        except Exception:
            pass

        return {
            "source": "booking.com",
            "hotel_name": hotel_name,
            "hotel_city": hotel_city,
            "reviewer_name": name,
            "nationality": nationality,
            "rating": rating,
            "review_title": title,
            "review_text": full,
            "positive_comment": pos,
            "negative_comment": neg,
            "publication_date": pub_date,
            "stay_date": stay_date,
            "trip_type": trip_type,
            "language": self.detect_language(full),
            **sub_scores,
        }

    def scrape_reviews(self, url, hotel_name, hotel_city, max_pages=90):
        print(f"🔄 Scraping : {hotel_name} ({hotel_city})")
        self.driver.get(url)
        time.sleep(6)

        all_reviews = []
        seen = set()
        page = 1

        while page <= max_pages:
            print(f"   📄 Page {page}  ({len(all_reviews)} avis cumulés)")

            try:
                self.wait.until(
                    EC.presence_of_element_located(
                        (By.CSS_SELECTOR, "div[data-testid='review-card']")
                    )
                )
            except Exception:
                print("   ⚠️  Aucune carte trouvée, fin du scraping.")
                break

            cards = self.driver.find_elements(
                By.CSS_SELECTOR, "div[data-testid='review-card']"
            )
            if not cards:
                print("   ⚠️  Liste de cartes vide, fin du scraping.")
                break

            snap = self._get_snapshot()

            new_count = 0
            for card in cards:
                review = self._parse_card(card, hotel_name, hotel_city)
                key = (
                    review.get("reviewer_name", ""),
                    review.get("publication_date", ""),
                    review.get("rating", ""),
                    review.get("review_text", "")[:50],
                )
                if key not in seen:
                    seen.add(key)
                    all_reviews.append(review)
                    new_count += 1

            print(f"      → {new_count} nouveaux avis")

            if new_count == 0:
                print("   ⚠️  Aucun nouvel avis — boucle détectée, arrêt.")
                break

            if not self._next_page(snap):
                print("   ✅ Dernière page atteinte.")
                break

            page += 1

        self.driver.quit()
        print(f"\n✅ {len(all_reviews)} avis extraits au total")
        return pd.DataFrame(all_reviews)


# ==================================================
if __name__ == "__main__":
    scraper = BookingScraper()
    df = scraper.scrape_reviews(URL, HOTEL_NAME, HOTEL_CITY, max_pages=MAX_PAGES)

    if not df.empty:
        filename = f"avis_{HOTEL_NAME.replace(' ', '_')}.csv"
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"\n💾 Fichier sauvegardé : {filename}  ({len(df)} avis)")

        print("\n🔍 Aperçu :")
        cols = [c for c in ["reviewer_name", "nationality", "rating",
                             "trip_type", "publication_date", "stay_date",
                             "language"] if c in df.columns]
        print(df[cols].head(10).to_string())

        print("\n📊 Colonnes vides :")
        empty = df[cols].isnull().sum() + (df[cols] == "").sum()
        print(empty[empty > 0] if empty[empty > 0].any() else "  Aucune colonne vide 🎉")
    else:
        print("❌ Aucun avis extrait.")

🔄 Scraping : Ibis Mohammedia (Mohammedia)
   📄 Page 1  (0 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 2  (10 avis cumulés)
      → 9 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 3  (19 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 4  (29 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 5  (39 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 6  (49 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 7  (59 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
 

In [17]:
import time
import re
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from lingua import Language, LanguageDetectorBuilder

# ==================================================
URL = "https://www.booking.com/hotel/ma/ibis-moussafir-ouarzazate.html?aid=356980&label=gog235jc-10CAsojAFCGWliaXMtbW91c3NhZmlyLW91YXJ6YXphdGVIDVgDaIwBiAEBmAEzuAEHyAEN2AED6AEB-AEBiAIBqAIBuAKBxKTPBsACAdICJDYyMTY5NWYzLWYxODctNDg2ZS04MWVjLTk4MTg2ODY0MzU3ZdgCAeACAQ&sid=4ba5cffa630e66002a1045ea6b678064&dist=0&keep_landing=1&sb_price_type=total&type=total&#tab-reviews"
HOTEL_NAME = "Ibis Ouarzazate"
HOTEL_CITY = "Ouarzazate"
MAX_PAGES = 190
# ==================================================


class BookingScraper:
    def __init__(self):
        options = Options()
        options.add_argument('--lang=en')
        options.add_argument('--disable-blink-features=AutomationControlled')
        options.add_experimental_option("excludeSwitches", ["enable-automation"])
        options.add_experimental_option('useAutomationExtension', False)
        options.add_argument(
            '--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
            'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
        )
        self.driver = webdriver.Chrome(
            service=Service(ChromeDriverManager().install()), options=options
        )
        self.wait = WebDriverWait(self.driver, 20)

        self.detector = LanguageDetectorBuilder.from_languages(
            Language.FRENCH, Language.ENGLISH, Language.GERMAN,
            Language.SPANISH, Language.ITALIAN, Language.PORTUGUESE,
            Language.DUTCH, Language.RUSSIAN, Language.ARABIC, Language.CHINESE,
        ).with_minimum_relative_distance(0.15).build()

        self._lang_map = {
            Language.FRENCH: "french", Language.ENGLISH: "english",
            Language.GERMAN: "german", Language.SPANISH: "spanish",
            Language.ITALIAN: "italian", Language.PORTUGUESE: "portuguese",
            Language.DUTCH: "dutch", Language.RUSSIAN: "russian",
            Language.ARABIC: "arabic", Language.CHINESE: "chinese",
        }

    def _text(self, parent, css):
        try:
            return parent.find_element(By.CSS_SELECTOR, css).text.strip()
        except Exception:
            return ""

    def _first_text(self, parent, *selectors):
        for sel in selectors:
            val = self._text(parent, sel)
            if val:
                return val
        return ""

    def _extract_number(self, raw):
        if not raw:
            return ""
        raw = raw.replace(",", ".")
        match = re.search(r'\d+(?:\.\d+)?', raw)
        return match.group(0) if match else ""

    def detect_language(self, text):
        if not text or len(text.strip()) < 5:
            return "unknown"
        try:
            lang = self.detector.detect_language_of(text)
            return self._lang_map.get(lang, "unknown") if lang else "unknown"
        except Exception:
            return "unknown"

    def _get_snapshot(self):
        try:
            card = self.driver.find_element(By.CSS_SELECTOR, "div[data-testid='review-card']")
            for sel in [
                "div[data-testid='review-positive-text']",
                "div[data-testid='review-negative-text']",
                "h4[data-testid='review-title']",
            ]:
                try:
                    t = card.find_element(By.CSS_SELECTOR, sel).text.strip()
                    if t:
                        return t
                except Exception:
                    pass
        except Exception:
            pass
        return ""

    def _next_page(self, snapshot_before):
        MAX_RETRIES = 3

        for attempt in range(1, MAX_RETRIES + 1):
            btn = None

            labels = ["Page suivante", "Suivant", "Next page", "Next"]
            for label in labels:
                try:
                    btn = self.driver.find_element(
                        By.CSS_SELECTOR, f"button[aria-label='{label}']"
                    )
                    print(f"      🔘 Bouton trouvé : '{label}'")
                    break
                except Exception:
                    pass

            if btn is None:
                try:
                    btn = self.driver.find_element(
                        By.XPATH,
                        "//button[contains(@aria-label,'suivante') or "
                        "contains(@aria-label,'Suivant') or "
                        "contains(@aria-label,'Next')]"
                    )
                    print(f"      🔘 Bouton via XPath : '{btn.get_attribute('aria-label')}'")
                except Exception:
                    pass

            if btn is None:
                print("   ⚠️  Bouton introuvable.")
                return False

            disabled      = btn.get_attribute("disabled")
            aria_disabled = btn.get_attribute("aria-disabled") or ""
            classes       = btn.get_attribute("class") or ""
            if disabled or aria_disabled == "true" or "disabled" in classes:
                print("   ✅ Bouton désactivé — dernière page atteinte.")
                return False

            try:
                self.driver.execute_script(
                    "arguments[0].scrollIntoView({block:'center'});", btn
                )
                time.sleep(1)
            except Exception:
                pass

            clicked = False
            for method in ['normal', 'js']:
                try:
                    if method == 'normal':
                        btn.click()
                    else:
                        self.driver.execute_script("arguments[0].click();", btn)
                    clicked = True
                    print(f"      ✅ Clic OK ({method}) — tentative {attempt}/{MAX_RETRIES}")
                    break
                except Exception as e:
                    print(f"      ⚠️  Clic {method} échoué : {e}")

            if not clicked:
                time.sleep(2)
                continue

            for _ in range(60):
                time.sleep(0.5)
                try:
                    cards = self.driver.find_elements(
                        By.CSS_SELECTOR, "div[data-testid='review-card']"
                    )
                    if not cards:
                        continue
                    first_text = ""
                    for sel in [
                        "div[data-testid='review-positive-text']",
                        "div[data-testid='review-negative-text']",
                        "h4[data-testid='review-title']",
                    ]:
                        try:
                            t = cards[0].find_element(By.CSS_SELECTOR, sel).text.strip()
                            if t:
                                first_text = t
                                break
                        except Exception:
                            pass
                    if first_text and first_text != snapshot_before:
                        return True
                except Exception:
                    pass

            print(f"   ⚠️  Contenu inchangé après 30s (tentative {attempt}/{MAX_RETRIES})")

            if attempt < MAX_RETRIES:
                print(f"   🔄 Nouvelle tentative dans 3s...")
                self.driver.execute_script("window.scrollTo(0, 0);")
                time.sleep(3)

        print("   ❌ Échec après 3 tentatives.")
        return False

    def _parse_card(self, card, hotel_name, hotel_city):
        name = self._first_text(
            card,
            "div.b08850ce41.f546354b44",
            "div[data-testid='reviewer-name'] span",
            "p.reviewer_name",
        )
        nationality = self._first_text(
            card,
            "span.d838fb5f41.aea5eccb71",
            "span[class*='reviewer_country']",
            "div[data-testid='reviewer-country'] span",
        )
        raw_rating = self._first_text(
            card,
            "div[data-testid='review-score'] div.bc946a29db",
            "div[data-testid='review-score']",
            "div.bui-review-score__badge",
        )
        rating    = self._extract_number(raw_rating)
        title     = self._first_text(card, "h4[data-testid='review-title']", "h3[class*='title']")
        pub_date  = self._first_text(card, "span[data-testid='review-date']", "p[data-testid='review-date']")
        pub_date  = re.sub(r'^Reviewed\s*:\s*', '', pub_date).strip()
        pub_date  = re.sub(r'^Rédigé le\s*:\s*', '', pub_date).strip()
        stay_date = self._first_text(card, "span[data-testid='review-stay-date']", "p.review_staydate")
        stay_date = re.sub(r'^Stayed\s*:\s*', '', stay_date).strip()
        stay_date = re.sub(r'^Séjour\s*:\s*', '', stay_date).strip()
        trip_type = self._first_text(
            card,
            "span[data-testid='review-traveler-type']",
            "p[data-testid='travel-type']",
            "div[data-testid='review-travel-type']",
        )
        pos  = self._text(card, "div[data-testid='review-positive-text']")
        neg  = self._text(card, "div[data-testid='review-negative-text']")
        full = (pos + " " + neg).strip()

        sub_scores = {}
        try:
            subs = card.find_elements(By.CSS_SELECTOR, "div[data-testid='review-subscore']")
            for s in subs:
                try:
                    cat = s.find_element(By.CSS_SELECTOR, "span").text.strip()
                    val = s.find_element(By.CSS_SELECTOR, "div.bc946a29db").text.strip()
                    sub_scores[cat] = self._extract_number(val)
                except Exception:
                    pass
        except Exception:
            pass

        return {
            "source": "booking.com",
            "hotel_name": hotel_name,
            "hotel_city": hotel_city,
            "reviewer_name": name,
            "nationality": nationality,
            "rating": rating,
            "review_title": title,
            "review_text": full,
            "positive_comment": pos,
            "negative_comment": neg,
            "publication_date": pub_date,
            "stay_date": stay_date,
            "trip_type": trip_type,
            "language": self.detect_language(full),
            **sub_scores,
        }

    def scrape_reviews(self, url, hotel_name, hotel_city, max_pages=90):
        print(f"🔄 Scraping : {hotel_name} ({hotel_city})")
        self.driver.get(url)
        time.sleep(6)

        all_reviews = []
        seen = set()
        page = 1

        while page <= max_pages:
            print(f"   📄 Page {page}  ({len(all_reviews)} avis cumulés)")

            try:
                self.wait.until(
                    EC.presence_of_element_located(
                        (By.CSS_SELECTOR, "div[data-testid='review-card']")
                    )
                )
            except Exception:
                print("   ⚠️  Aucune carte trouvée, fin du scraping.")
                break

            cards = self.driver.find_elements(
                By.CSS_SELECTOR, "div[data-testid='review-card']"
            )
            if not cards:
                print("   ⚠️  Liste de cartes vide, fin du scraping.")
                break

            snap = self._get_snapshot()

            new_count = 0
            for card in cards:
                review = self._parse_card(card, hotel_name, hotel_city)
                key = (
                    review.get("reviewer_name", ""),
                    review.get("publication_date", ""),
                    review.get("rating", ""),
                    review.get("review_text", "")[:50],
                )
                if key not in seen:
                    seen.add(key)
                    all_reviews.append(review)
                    new_count += 1

            print(f"      → {new_count} nouveaux avis")

            if new_count == 0:
                print("   ⚠️  Aucun nouvel avis — boucle détectée, arrêt.")
                break

            if not self._next_page(snap):
                print("   ✅ Dernière page atteinte.")
                break

            page += 1

        self.driver.quit()
        print(f"\n✅ {len(all_reviews)} avis extraits au total")
        return pd.DataFrame(all_reviews)


# ==================================================
if __name__ == "__main__":
    scraper = BookingScraper()
    df = scraper.scrape_reviews(URL, HOTEL_NAME, HOTEL_CITY, max_pages=MAX_PAGES)

    if not df.empty:
        filename = f"avis_{HOTEL_NAME.replace(' ', '_')}.csv"
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"\n💾 Fichier sauvegardé : {filename}  ({len(df)} avis)")

        print("\n🔍 Aperçu :")
        cols = [c for c in ["reviewer_name", "nationality", "rating",
                             "trip_type", "publication_date", "stay_date",
                             "language"] if c in df.columns]
        print(df[cols].head(10).to_string())

        print("\n📊 Colonnes vides :")
        empty = df[cols].isnull().sum() + (df[cols] == "").sum()
        print(empty[empty > 0] if empty[empty > 0].any() else "  Aucune colonne vide 🎉")
    else:
        print("❌ Aucun avis extrait.")

🔄 Scraping : Ibis Ouarzazate (Ouarzazate)
   📄 Page 1  (0 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 2  (10 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 3  (20 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 4  (30 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 5  (40 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 6  (50 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 7  (60 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3


In [18]:
import time
import re
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from lingua import Language, LanguageDetectorBuilder

# ==================================================
URL = "https://www.booking.com/hotel/ma/ibis-moussafir-el-jadida.html?aid=356980&label=gog235jc-10CAsojAFCGGliaXMtbW91c3NhZmlyLWVsLWphZGlkYUgNWANojAGIAQGYATO4AQfIAQ3YAQPoAQH4AQGIAgGoAgG4As3KpM8GwAIB0gIkZjI4MDVkM2UtMjI4Yi00OWE2LTlhMzItYzc4ZGQyMWZlODY52AIB4AIB&sid=4ba5cffa630e66002a1045ea6b678064&dist=0&keep_landing=1&sb_price_type=total&type=total&#tab-reviews"
HOTEL_NAME = "Ibis El Jadida"
HOTEL_CITY = "El Jadida"
MAX_PAGES = 90
# ==================================================


class BookingScraper:
    def __init__(self):
        options = Options()
        options.add_argument('--lang=en')
        options.add_argument('--disable-blink-features=AutomationControlled')
        options.add_experimental_option("excludeSwitches", ["enable-automation"])
        options.add_experimental_option('useAutomationExtension', False)
        options.add_argument(
            '--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
            'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
        )
        self.driver = webdriver.Chrome(
            service=Service(ChromeDriverManager().install()), options=options
        )
        self.wait = WebDriverWait(self.driver, 20)

        self.detector = LanguageDetectorBuilder.from_languages(
            Language.FRENCH, Language.ENGLISH, Language.GERMAN,
            Language.SPANISH, Language.ITALIAN, Language.PORTUGUESE,
            Language.DUTCH, Language.RUSSIAN, Language.ARABIC, Language.CHINESE,
        ).with_minimum_relative_distance(0.15).build()

        self._lang_map = {
            Language.FRENCH: "french", Language.ENGLISH: "english",
            Language.GERMAN: "german", Language.SPANISH: "spanish",
            Language.ITALIAN: "italian", Language.PORTUGUESE: "portuguese",
            Language.DUTCH: "dutch", Language.RUSSIAN: "russian",
            Language.ARABIC: "arabic", Language.CHINESE: "chinese",
        }

    def _text(self, parent, css):
        try:
            return parent.find_element(By.CSS_SELECTOR, css).text.strip()
        except Exception:
            return ""

    def _first_text(self, parent, *selectors):
        for sel in selectors:
            val = self._text(parent, sel)
            if val:
                return val
        return ""

    def _extract_number(self, raw):
        if not raw:
            return ""
        raw = raw.replace(",", ".")
        match = re.search(r'\d+(?:\.\d+)?', raw)
        return match.group(0) if match else ""

    def detect_language(self, text):
        if not text or len(text.strip()) < 5:
            return "unknown"
        try:
            lang = self.detector.detect_language_of(text)
            return self._lang_map.get(lang, "unknown") if lang else "unknown"
        except Exception:
            return "unknown"

    def _get_snapshot(self):
        try:
            card = self.driver.find_element(By.CSS_SELECTOR, "div[data-testid='review-card']")
            for sel in [
                "div[data-testid='review-positive-text']",
                "div[data-testid='review-negative-text']",
                "h4[data-testid='review-title']",
            ]:
                try:
                    t = card.find_element(By.CSS_SELECTOR, sel).text.strip()
                    if t:
                        return t
                except Exception:
                    pass
        except Exception:
            pass
        return ""

    def _next_page(self, snapshot_before):
        MAX_RETRIES = 3

        for attempt in range(1, MAX_RETRIES + 1):
            btn = None

            labels = ["Page suivante", "Suivant", "Next page", "Next"]
            for label in labels:
                try:
                    btn = self.driver.find_element(
                        By.CSS_SELECTOR, f"button[aria-label='{label}']"
                    )
                    print(f"      🔘 Bouton trouvé : '{label}'")
                    break
                except Exception:
                    pass

            if btn is None:
                try:
                    btn = self.driver.find_element(
                        By.XPATH,
                        "//button[contains(@aria-label,'suivante') or "
                        "contains(@aria-label,'Suivant') or "
                        "contains(@aria-label,'Next')]"
                    )
                    print(f"      🔘 Bouton via XPath : '{btn.get_attribute('aria-label')}'")
                except Exception:
                    pass

            if btn is None:
                print("   ⚠️  Bouton introuvable.")
                return False

            disabled      = btn.get_attribute("disabled")
            aria_disabled = btn.get_attribute("aria-disabled") or ""
            classes       = btn.get_attribute("class") or ""
            if disabled or aria_disabled == "true" or "disabled" in classes:
                print("   ✅ Bouton désactivé — dernière page atteinte.")
                return False

            try:
                self.driver.execute_script(
                    "arguments[0].scrollIntoView({block:'center'});", btn
                )
                time.sleep(1)
            except Exception:
                pass

            clicked = False
            for method in ['normal', 'js']:
                try:
                    if method == 'normal':
                        btn.click()
                    else:
                        self.driver.execute_script("arguments[0].click();", btn)
                    clicked = True
                    print(f"      ✅ Clic OK ({method}) — tentative {attempt}/{MAX_RETRIES}")
                    break
                except Exception as e:
                    print(f"      ⚠️  Clic {method} échoué : {e}")

            if not clicked:
                time.sleep(2)
                continue

            for _ in range(60):
                time.sleep(0.5)
                try:
                    cards = self.driver.find_elements(
                        By.CSS_SELECTOR, "div[data-testid='review-card']"
                    )
                    if not cards:
                        continue
                    first_text = ""
                    for sel in [
                        "div[data-testid='review-positive-text']",
                        "div[data-testid='review-negative-text']",
                        "h4[data-testid='review-title']",
                    ]:
                        try:
                            t = cards[0].find_element(By.CSS_SELECTOR, sel).text.strip()
                            if t:
                                first_text = t
                                break
                        except Exception:
                            pass
                    if first_text and first_text != snapshot_before:
                        return True
                except Exception:
                    pass

            print(f"   ⚠️  Contenu inchangé après 30s (tentative {attempt}/{MAX_RETRIES})")

            if attempt < MAX_RETRIES:
                print(f"   🔄 Nouvelle tentative dans 3s...")
                self.driver.execute_script("window.scrollTo(0, 0);")
                time.sleep(3)

        print("   ❌ Échec après 3 tentatives.")
        return False

    def _parse_card(self, card, hotel_name, hotel_city):
        name = self._first_text(
            card,
            "div.b08850ce41.f546354b44",
            "div[data-testid='reviewer-name'] span",
            "p.reviewer_name",
        )
        nationality = self._first_text(
            card,
            "span.d838fb5f41.aea5eccb71",
            "span[class*='reviewer_country']",
            "div[data-testid='reviewer-country'] span",
        )
        raw_rating = self._first_text(
            card,
            "div[data-testid='review-score'] div.bc946a29db",
            "div[data-testid='review-score']",
            "div.bui-review-score__badge",
        )
        rating    = self._extract_number(raw_rating)
        title     = self._first_text(card, "h4[data-testid='review-title']", "h3[class*='title']")
        pub_date  = self._first_text(card, "span[data-testid='review-date']", "p[data-testid='review-date']")
        pub_date  = re.sub(r'^Reviewed\s*:\s*', '', pub_date).strip()
        pub_date  = re.sub(r'^Rédigé le\s*:\s*', '', pub_date).strip()
        stay_date = self._first_text(card, "span[data-testid='review-stay-date']", "p.review_staydate")
        stay_date = re.sub(r'^Stayed\s*:\s*', '', stay_date).strip()
        stay_date = re.sub(r'^Séjour\s*:\s*', '', stay_date).strip()
        trip_type = self._first_text(
            card,
            "span[data-testid='review-traveler-type']",
            "p[data-testid='travel-type']",
            "div[data-testid='review-travel-type']",
        )
        pos  = self._text(card, "div[data-testid='review-positive-text']")
        neg  = self._text(card, "div[data-testid='review-negative-text']")
        full = (pos + " " + neg).strip()

        sub_scores = {}
        try:
            subs = card.find_elements(By.CSS_SELECTOR, "div[data-testid='review-subscore']")
            for s in subs:
                try:
                    cat = s.find_element(By.CSS_SELECTOR, "span").text.strip()
                    val = s.find_element(By.CSS_SELECTOR, "div.bc946a29db").text.strip()
                    sub_scores[cat] = self._extract_number(val)
                except Exception:
                    pass
        except Exception:
            pass

        return {
            "source": "booking.com",
            "hotel_name": hotel_name,
            "hotel_city": hotel_city,
            "reviewer_name": name,
            "nationality": nationality,
            "rating": rating,
            "review_title": title,
            "review_text": full,
            "positive_comment": pos,
            "negative_comment": neg,
            "publication_date": pub_date,
            "stay_date": stay_date,
            "trip_type": trip_type,
            "language": self.detect_language(full),
            **sub_scores,
        }

    def scrape_reviews(self, url, hotel_name, hotel_city, max_pages=90):
        print(f"🔄 Scraping : {hotel_name} ({hotel_city})")
        self.driver.get(url)
        time.sleep(6)

        all_reviews = []
        seen = set()
        page = 1

        while page <= max_pages:
            print(f"   📄 Page {page}  ({len(all_reviews)} avis cumulés)")

            try:
                self.wait.until(
                    EC.presence_of_element_located(
                        (By.CSS_SELECTOR, "div[data-testid='review-card']")
                    )
                )
            except Exception:
                print("   ⚠️  Aucune carte trouvée, fin du scraping.")
                break

            cards = self.driver.find_elements(
                By.CSS_SELECTOR, "div[data-testid='review-card']"
            )
            if not cards:
                print("   ⚠️  Liste de cartes vide, fin du scraping.")
                break

            snap = self._get_snapshot()

            new_count = 0
            for card in cards:
                review = self._parse_card(card, hotel_name, hotel_city)
                key = (
                    review.get("reviewer_name", ""),
                    review.get("publication_date", ""),
                    review.get("rating", ""),
                    review.get("review_text", "")[:50],
                )
                if key not in seen:
                    seen.add(key)
                    all_reviews.append(review)
                    new_count += 1

            print(f"      → {new_count} nouveaux avis")

            if new_count == 0:
                print("   ⚠️  Aucun nouvel avis — boucle détectée, arrêt.")
                break

            if not self._next_page(snap):
                print("   ✅ Dernière page atteinte.")
                break

            page += 1

        self.driver.quit()
        print(f"\n✅ {len(all_reviews)} avis extraits au total")
        return pd.DataFrame(all_reviews)


# ==================================================
if __name__ == "__main__":
    scraper = BookingScraper()
    df = scraper.scrape_reviews(URL, HOTEL_NAME, HOTEL_CITY, max_pages=MAX_PAGES)

    if not df.empty:
        filename = f"avis_{HOTEL_NAME.replace(' ', '_')}.csv"
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"\n💾 Fichier sauvegardé : {filename}  ({len(df)} avis)")

        print("\n🔍 Aperçu :")
        cols = [c for c in ["reviewer_name", "nationality", "rating",
                             "trip_type", "publication_date", "stay_date",
                             "language"] if c in df.columns]
        print(df[cols].head(10).to_string())

        print("\n📊 Colonnes vides :")
        empty = df[cols].isnull().sum() + (df[cols] == "").sum()
        print(empty[empty > 0] if empty[empty > 0].any() else "  Aucune colonne vide 🎉")
    else:
        print("❌ Aucun avis extrait.")

🔄 Scraping : Ibis El Jadida (El Jadida)
   📄 Page 1  (0 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 2  (10 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 3  (20 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 4  (30 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 5  (40 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 6  (50 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
   📄 Page 7  (60 avis cumulés)
      → 10 nouveaux avis
      🔘 Bouton trouvé : 'Page suivante'
      ✅ Clic OK (normal) — tentative 1/3
  

In [22]:
df.columns

Index(['source', 'hotel_name', 'hotel_city', 'reviewer_name', 'nationality',
       'rating', 'review_title', 'review_text', 'positive_comment',
       'negative_comment', 'publication_date', 'stay_date', 'trip_type',
       'language'],
      dtype='str')

In [1]:
import pandas as pd

files = [
    "avis_Ibis_Casablanca_Abdelmoumen_booking.csv",
    "avis_Ibis_Casablanca_City_Center_booking.csv",
    "avis_Ibis_Casablanca_Nearshore_booking.csv",
    "avis_Ibis_Casablanca_Sidi_Maarouf_booking.csv",
    "avis_Ibis_Casablanca_Voyageurs_booking.csv",
    "avis_Ibis_El_Jadida_booking.csv",
    "avis_Ibis_Marrakech_Centre_Gare_booking.csv",
    "avis_Ibis_Marrakech_Palmeraie_booking.csv",
    "avis_Ibis_Meknes_booking.csv",
    "avis_Ibis_Mohammedia_booking.csv",
    "avis_Ibis_Moussafir_Fès_booking.csv",
    "avis_Ibis_Ouarzazate_booking.csv",
    "avis_Ibis_Oujda_booking.csv",
    "avis_Ibis_Rabat_Agdal_booking.csv",
    "avis_Ibis_Tanger_City_Center_booking.csv",
]

Avis_booking_total = pd.concat(
    [pd.read_csv(f) for f in files], ignore_index=True
)

print(f"Shape : {Avis_booking_total.shape}")
print(Avis_booking_total.head())

Avis_booking_total.to_csv("Avis_booking_total.csv", index=False)
print("Sauvegardé ✓")

Shape : (14771, 14)
        source                   hotel_name  hotel_city reviewer_name  \
0  booking.com  Ibis Casablanca Abdelmoumen  Casablanca         Edris   
1  booking.com  Ibis Casablanca Abdelmoumen  Casablanca        Samata   
2  booking.com  Ibis Casablanca Abdelmoumen  Casablanca        Tasnim   
3  booking.com  Ibis Casablanca Abdelmoumen  Casablanca         Asmaa   
4  booking.com  Ibis Casablanca Abdelmoumen  Casablanca          Adda   

  nationality  rating                                       review_title  \
0       Maroc     7.0                                               Bien   
1      France     7.0                                               Bien   
2      France     8.0                     Excellent rapport qualité prix   
3       Maroc     7.0  Un personnel agréable et un très bn rapport qu...   
4       Maroc     8.0                                              حميلة   

                                         review_text  \
0                           

In [2]:
Avis_booking_total.head()

,source,hotel_name,hotel_city,reviewer_name,nationality,rating,review_title,review_text,positive_comment,negative_comment,publication_date,stay_date,trip_type,language
0,booking.com,Ibis Casablanca Abdelmoumen,Casablanca,Edris,Maroc,7.0,Bien,Tout Bruit,Tout,Bruit,Commentaire envoyé le 22 avril 2026,avril 2026,Voyageur individuel,french
1,booking.com,Ibis Casablanca Abdelmoumen,Casablanca,Samata,France,7.0,Bien,"Hôtel à proximité vraiment de tout , propre et...","Hôtel à proximité vraiment de tout , propre et...",Dommage le lit est dure !!!!!!!,Commentaire envoyé le 14 avril 2026,avril 2026,Famille,french
2,booking.com,Ibis Casablanca Abdelmoumen,Casablanca,Tasnim,France,8.0,Excellent rapport qualité prix,"Chambre spacieuse, hôtel moderne et personnel ...","Chambre spacieuse, hôtel moderne et personnel ...",NaN,Commentaire envoyé le 11 avril 2026,avril 2026,Voyageur individuel,french
3,booking.com,Ibis Casablanca Abdelmoumen,Casablanca,Asmaa,Maroc,7.0,Un personnel agréable et un très bn rapport qu...,Situation géographique Il y a du bruit vu just...,Situation géographique,Il y a du bruit vu justement l'emplacement,Commentaire envoyé le 5 avril 2026,mars 2026,Voyageur individuel,french
4,booking.com,Ibis Casablanca Abdelmoumen,Casablanca,Adda,Maroc,8.0,حميلة,نعن تمن مرتفع,نعن,تمن مرتفع,Commentaire envoyé le 29 mars 2026,mars 2026,Voyageur individuel,arabic


In [3]:
Avis_booking_total.columns

Index(['source', 'hotel_name', 'hotel_city', 'reviewer_name', 'nationality',
       'rating', 'review_title', 'review_text', 'positive_comment',
       'negative_comment', 'publication_date', 'stay_date', 'trip_type',
       'language'],
      dtype='str')